# 미국 언론사 RSS 과거 뉴스 수집 테스트

검색어 없이 **언론사가 제공하는 RSS 피드만으로** 과거 특정 기간에 나온 기사 목록을 전부 모을 수 있는지 피드별로 테스트하는 노트북입니다.
일반 RSS는 최신 기사 10~100건만 보여주기 때문에, 과거로 거슬러 가는 방법이 있는 피드만 쓸 수 있습니다.

**과거 접근 방식 (우선순위 순)**

| 방식 | 원리 | 장점 / 한계 |
|---|---|---|
| `DATE_ARCHIVE` | WordPress 날짜 아카이브 피드 `/{YYYY}/{MM}/{DD}/feed/?paged=N` | 날짜를 직접 지정, 그날 기사를 끝 페이지까지 전부. 사이트 전체 피드만 가능 |
| `PAGED` | 최신순 피드에 `?paged=N`(또는 `page=N`)을 붙여 과거 페이지로 이동 | 목표 기간이 있는 페이지를 이진탐색. 페이지 상한까지만 내려감 |

**수집에서 제외한 방식**: archive.org(Wayback Machine)에 저장된 피드 스냅샷 이어붙이기.
CNBC·WSJ·MarketWatch·NYT 등은 2025-10-02 무렵부터 15분 간격 스냅샷이 있어 이론상 가능하지만,
스냅샷 본문 요청이 Python에서는 4초 간격 + 휴식 후에도 대부분 HTTP 429로 막혀서 제외했습니다 (2026-11-04 테스트).
2-3단계에서 스냅샷 유무만 참고용으로 조회해 `WAYBACK_ONLY_FEEDS`에 남깁니다.

**단계**
1. 라이브 피드 점검 (응답, 기사 수, 날짜 파싱, 갱신 여부)
2. 과거 접근 방식 판별 + 깊이 테스트 (어디까지 과거로 내려가는지)
3. 테스트 기간 실제 수집
4. 판정 → 피드 딕셔너리 정리 (`PASSED_*`, `PARTIAL_FEEDS`, `LIVE_ONLY_FEEDS`, `WAYBACK_ONLY_FEEDS`, `FAILED_FEEDS`)
5. 실제 수집용 함수 `collect_news(start, end)` (기사 목록)
6. 기사 목록 눈으로 검수
7. 본문 수집 테스트: 통과 피드의 표본 기사로 RSS 본문과 원문 페이지 본문을 각각 추출하고 서로 대조해서 검증
   → 본문 딕셔너리 정리 (`RSS_BODY_FEEDS`, `PAGE_BODY_FEEDS`, `BODY_UNSTABLE_FEEDS`, `BODY_NOT_USABLE_FEEDS`)
8. 본문까지 붙이는 수집 함수 `collect_news_with_body(start, end)`
9. 본문 눈으로 검수

**캐시**: 과거 날짜 아카이브 피드, Wayback CDX 조회 결과는 바뀌지 않으므로 `rss_history/cache/`에 저장하고 재요청하지 않습니다.
라이브 피드와 `PAGED` 페이지는 새 기사가 올라오면 내용이 밀리므로 캐시하지 않습니다.
기사 원문 HTML도 `rss_history/cache/html/`에 저장해서, 7단계 판정 기준만 바꿔 다시 돌릴 때는 네트워크 요청이 없습니다.

**참고한 코드**: 원문 본문 추출·페이지 판정은 `1_google_rss_source_test.ipynb`(trafilatura + JSON-LD + 차단/유료/영상 판정),
RSS 본문 처리와 검증 항목 일부는 `rss_news/` 폴더(팀원 작성)의 본문 시험 코드와 매체 검토 문서를 참고했습니다.

In [1]:
import calendar
import gzip
import hashlib
import html
import json
import re
import time
import unicodedata
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests
from dateutil import parser as date_parser
from lxml import html as lxml_html
from trafilatura import extract as trafilatura_extract
from trafilatura.utils import decode_file

## 0. 설정

In [2]:
# ============================================================
# 테스트 기간 (미국 동부시간 날짜, 양 끝 포함)
# ============================================================
# 약 1년 전 평일 이틀. 다른 시점은 아래 DEPTH_PROBE_DATES로 확인
TEST_START = date(2025, 10, 14)
TEST_END = date(2025, 10, 15)

# 과거 어디까지 내려갈 수 있는지 확인할 날짜 (깊이 테스트)
DEPTH_PROBE_DATES = [
    date(2025, 6, 10),
    date(2024, 10, 15),
    date(2023, 10, 17),
    date(2022, 10, 18),
]

NEWS_TIMEZONE = ZoneInfo("America/New_York")

# 최신 기사가 이보다 오래됐으면 갱신이 멈춘 피드(STALE)
STALE_DAYS = 7

# 날짜가 있는 기사 비율이 이보다 낮으면 NO_DATE
MIN_DATE_RATE = 0.8

# 날짜 아카이브: 요청한 날짜(±1일)에 속한 기사 비율이 이 이상이어야 통과
MIN_ARCHIVE_MATCH_RATE = 0.8

# 무한 루프 방지용 상한
MAX_PAGES_PER_DAY = 60
MAX_PAGE_NUMBER = 50000

REQUEST_TIMEOUT = 30
REQUEST_DELAY = 0.5     # 같은 호스트 연속 요청 간격(초)
MAX_RETRIES = 3

# archive.org CDX 조회 간격(초). 429를 받으면 WAYBACK_BLOCK_WAIT × 시도 횟수만큼 쉬고 재시도
WAYBACK_DELAY = 2.0
WAYBACK_BLOCK_WAIT = 60

# 이보다 과거 날짜는 깨진 값으로 간주 (예: Fast Company 옛 기사의 1969-12-31)
MIN_VALID_DATE = datetime(1995, 1, 1, tzinfo=timezone.utc)

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36"
    ),
    "Accept": "application/rss+xml, application/atom+xml, application/xml;q=0.9, */*;q=0.8",
}

BASE_DIR = Path("rss_history")
CACHE_DIR = BASE_DIR / "cache"
FEED_CACHE_DIR = CACHE_DIR / "feeds"
CDX_CACHE_PATH = CACHE_DIR / "wayback_cdx.jsonl"

LIVE_RESULT_PATH = BASE_DIR / "live_check.csv"
ACCESS_RESULT_PATH = BASE_DIR / "access_check.csv"
ARTICLES_PATH = BASE_DIR / f"articles_{TEST_START}_{TEST_END}.parquet"
SUMMARY_PATH = BASE_DIR / "feed_summary.csv"
FEED_LISTS_PATH = BASE_DIR / "feed_lists.json"

In [3]:
# ============================================================
# 테스트 후보 피드
#   feed_id: {source: 도메인, name: 언론사명, section: 섹션, url: 피드 URL}
#
# - 언론사 하나가 섹션 피드를 여러 개 제공하면 섹션별로 따로 둡니다.
# - WordPress 사이트는 사이트 전체 피드(/feed/)여야 날짜 아카이브 테스트 대상이 됩니다.
# - 2026-11-04 사전 조사에서 이미 응답이 없던 피드도 기록을 위해 포함 (1단계에서 걸러짐)
# ============================================================
def feed(source, name, section, url):
    return {"source": source, "name": name, "section": section, "url": url}


CNBC = "https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id="
DJ = "https://feeds.content.dowjones.io/public/rss/"

CANDIDATE_FEEDS = {
    # ── 미국 증시/금융 전문 ──
    "cnbc_top": feed("cnbc.com", "CNBC", "Top News", CNBC + "100003114"),
    "cnbc_business": feed("cnbc.com", "CNBC", "Business", CNBC + "10001147"),
    "cnbc_investing": feed("cnbc.com", "CNBC", "Investing", CNBC + "15839069"),
    "cnbc_market_insider": feed("cnbc.com", "CNBC", "Market Insider", CNBC + "20409666"),
    "cnbc_economy": feed("cnbc.com", "CNBC", "Economy", CNBC + "20910258"),
    "cnbc_earnings": feed("cnbc.com", "CNBC", "Earnings", CNBC + "15839135"),
    "cnbc_finance": feed("cnbc.com", "CNBC", "Finance", CNBC + "10000664"),
    "cnbc_tech": feed("cnbc.com", "CNBC", "Tech", CNBC + "19854910"),

    "wsj_markets": feed("wsj.com", "Wall Street Journal", "Markets", DJ + "RSSMarketsMain"),
    "wsj_business": feed("wsj.com", "Wall Street Journal", "US Business", DJ + "WSJcomUSBusiness"),
    "wsj_economy": feed("wsj.com", "Wall Street Journal", "Economy", DJ + "socialeconomyfeed"),
    "wsj_tech": feed("wsj.com", "Wall Street Journal", "Technology", DJ + "RSSWSJD"),
    "wsj_world": feed("wsj.com", "Wall Street Journal", "World", DJ + "RSSWorldNews"),

    "marketwatch_top": feed("marketwatch.com", "MarketWatch", "Top Stories", DJ + "mw_topstories"),
    "marketwatch_bulletins": feed("marketwatch.com", "MarketWatch", "Breaking News", DJ + "mw_bulletins"),
    "marketwatch_pulse": feed("marketwatch.com", "MarketWatch", "MarketPulse", DJ + "mw_marketpulse"),

    "barrons": feed("barrons.com", "Barron's", "All", "https://www.barrons.com/feed"),
    "bloomberg_markets": feed("bloomberg.com", "Bloomberg", "Markets", "https://feeds.bloomberg.com/markets/news.rss"),
    "bloomberg_economics": feed("bloomberg.com", "Bloomberg", "Economics", "https://feeds.bloomberg.com/economics/news.rss"),
    "bloomberg_tech": feed("bloomberg.com", "Bloomberg", "Technology", "https://feeds.bloomberg.com/technology/news.rss"),

    "ibd": feed("investors.com", "Investor's Business Daily", "All", "https://www.investors.com/feed/"),
    "seekingalpha_all": feed("seekingalpha.com", "Seeking Alpha", "All Articles", "https://seekingalpha.com/feed.xml"),
    "seekingalpha_currents": feed("seekingalpha.com", "Seeking Alpha", "Market Currents", "https://seekingalpha.com/market_currents.xml"),
    "fool": feed("fool.com", "The Motley Fool", "All", "https://www.fool.com/a/feeds/foolwatch?apikey=foolwatch-feed"),
    "nasdaq_markets": feed("nasdaq.com", "Nasdaq", "Markets", "https://www.nasdaq.com/feed/rssoutbound?category=Markets"),
    "investing_news": feed("investing.com", "Investing.com", "All News", "https://www.investing.com/rss/news.rss"),
    "investing_economy": feed("investing.com", "Investing.com", "Economy", "https://www.investing.com/rss/news_14.rss"),
    "marketbeat": feed("marketbeat.com", "MarketBeat", "All", "https://www.marketbeat.com/feed/"),
    "thestreet": feed("thestreet.com", "TheStreet", "All", "https://www.thestreet.com/.rss/full/"),
    "kiplinger": feed("kiplinger.com", "Kiplinger", "All", "https://www.kiplinger.com/feeds.xml"),
    "yahoo_finance": feed("finance.yahoo.com", "Yahoo Finance", "All", "https://finance.yahoo.com/news/rssindex"),
    "zacks": feed("zacks.com", "Zacks", "All", "https://scr.zacks.com/rss"),

    # ── 경제/비즈니스 종합 ──
    "fortune": feed("fortune.com", "Fortune", "All", "https://fortune.com/feed/"),
    "business_insider": feed("businessinsider.com", "Business Insider", "All", "https://feeds.businessinsider.com/custom/all"),
    "foxbusiness_latest": feed("foxbusiness.com", "Fox Business", "Latest", "https://moxie.foxbusiness.com/google-publisher/latest.xml"),
    "foxbusiness_markets": feed("foxbusiness.com", "Fox Business", "Markets", "https://moxie.foxbusiness.com/google-publisher/markets.xml"),
    "foxbusiness_economy": feed("foxbusiness.com", "Fox Business", "Economy", "https://moxie.foxbusiness.com/google-publisher/economy.xml"),
    "cbs_moneywatch": feed("cbsnews.com", "CBS News", "MoneyWatch", "https://www.cbsnews.com/latest/rss/moneywatch"),
    "nypost": feed("nypost.com", "New York Post", "All", "https://nypost.com/feed/"),
    "nypost_business": feed("nypost.com", "New York Post", "Business", "https://nypost.com/business/feed/"),
    "usatoday_money": feed("usatoday.com", "USA Today", "Money", "https://rssfeeds.usatoday.com/UsatodaycomMoney-TopStories"),
    "cnn_money": feed("cnn.com", "CNN Business", "Money", "http://rss.cnn.com/rss/money_latest.rss"),
    "forbes_business": feed("forbes.com", "Forbes", "Business", "https://www.forbes.com/business/feed/"),
    "forbes_innovation": feed("forbes.com", "Forbes", "Innovation", "https://www.forbes.com/innovation/feed/"),
    "fastcompany": feed("fastcompany.com", "Fast Company", "Latest", "https://www.fastcompany.com/latest/rss"),
    "qz": feed("qz.com", "Quartz", "All", "https://qz.com/rss"),
    "axios": feed("axios.com", "Axios", "All", "https://api.axios.com/feed/"),
    "semafor": feed("semafor.com", "Semafor", "All", "https://www.semafor.com/rss.xml"),

    # ── 종합 일간지/방송 ──
    "nyt_business": feed("nytimes.com", "New York Times", "Business", "https://rss.nytimes.com/services/xml/rss/nyt/Business.xml"),
    "nyt_economy": feed("nytimes.com", "New York Times", "Economy", "https://rss.nytimes.com/services/xml/rss/nyt/Economy.xml"),
    "nyt_tech": feed("nytimes.com", "New York Times", "Technology", "https://rss.nytimes.com/services/xml/rss/nyt/Technology.xml"),
    "nyt_home": feed("nytimes.com", "New York Times", "Home Page", "https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml"),
    "wapo_business": feed("washingtonpost.com", "Washington Post", "Business", "https://feeds.washingtonpost.com/rss/business"),
    "latimes_business": feed("latimes.com", "Los Angeles Times", "Business", "https://www.latimes.com/business/rss2.0.xml"),
    "guardian_us_business": feed("theguardian.com", "The Guardian", "US Business", "https://www.theguardian.com/us/business/rss"),
    "npr_business": feed("npr.org", "NPR", "Business", "https://feeds.npr.org/1006/rss.xml"),
    "npr_economy": feed("npr.org", "NPR", "Economy", "https://feeds.npr.org/1017/rss.xml"),
    "abc_business": feed("abcnews.go.com", "ABC News", "Business", "https://abcnews.go.com/abcnews/moneyheadlines"),
    "nbc_business": feed("nbcnews.com", "NBC News", "Business", "https://feeds.nbcnews.com/nbcnews/public/business"),
    "apnews": feed("apnews.com", "Associated Press", "All", "https://apnews.com/index.rss"),
    "yahoo_news": feed("yahoo.com", "Yahoo News", "All", "https://news.yahoo.com/rss/"),
    "politico": feed("politico.com", "Politico", "Politics", "https://rss.politico.com/politics-news.xml"),
    "thehill": feed("thehill.com", "The Hill", "All", "https://thehill.com/feed/"),
    "time": feed("time.com", "TIME", "All", "https://time.com/feed/"),
    "washington_examiner": feed("washingtonexaminer.com", "Washington Examiner", "All", "https://www.washingtonexaminer.com/feed/"),
    "nationalreview": feed("nationalreview.com", "National Review", "All", "https://www.nationalreview.com/feed/"),
    "motherjones": feed("motherjones.com", "Mother Jones", "All", "https://www.motherjones.com/feed/"),

    # ── IT/반도체 ──
    "techcrunch": feed("techcrunch.com", "TechCrunch", "All", "https://techcrunch.com/feed/"),
    "theverge": feed("theverge.com", "The Verge", "All", "https://www.theverge.com/rss/index.xml"),
    "arstechnica": feed("arstechnica.com", "Ars Technica", "All", "https://arstechnica.com/feed/"),
    "wired": feed("wired.com", "WIRED", "All", "https://www.wired.com/feed/rss"),
    "engadget": feed("engadget.com", "Engadget", "All", "https://www.engadget.com/rss.xml"),
    "cnet": feed("cnet.com", "CNET", "News", "https://www.cnet.com/rss/news/"),
    "9to5mac": feed("9to5mac.com", "9to5Mac", "All", "https://9to5mac.com/feed/"),
    "tomshardware": feed("tomshardware.com", "Tom's Hardware", "All", "https://www.tomshardware.com/feeds.xml"),
    "eetimes": feed("eetimes.com", "EE Times", "All", "https://www.eetimes.com/feed/"),
    "semiengineering": feed("semiengineering.com", "Semiconductor Engineering", "All", "https://semiengineering.com/feed/"),
    "venturebeat": feed("venturebeat.com", "VentureBeat", "All", "https://venturebeat.com/feed/"),

    # ── 보도자료 배포 ──
    "businesswire": feed("businesswire.com", "Business Wire", "All", "https://feed.businesswire.com/rss/home/?rss=G1QFDERJXkJeEFtRWA=="),
    "prnewswire": feed("prnewswire.com", "PR Newswire", "All", "https://www.prnewswire.com/rss/news-releases-list.rss"),
    "globenewswire": feed("globenewswire.com", "GlobeNewswire", "Public Companies", "https://www.globenewswire.com/RssFeed/orgclass/1/feedTitle/GlobeNewswire%20-%20News%20about%20Public%20Companies"),
}

# 테스트 전에 내용을 보고 제외한 피드와 이유
EXCLUDED_FEEDS = {
    # /feed 가 뉴스가 아니라 재테크 가이드(benzinga.com/money) 글 목록이고,
    # 1페이지 기사 시각이 1초 간격으로 일괄 갱신돼 있어 작성 시각으로 쓸 수 없음
    "benzinga": "재테크 가이드 피드 (뉴스 아님, 작성 시각 신뢰 불가)",
}

print(f"후보 피드 {len(CANDIDATE_FEEDS)}개 / 언론사 {len({f['source'] for f in CANDIDATE_FEEDS.values()})}곳")

후보 피드 81개 / 언론사 56곳


In [4]:
# ============================================================
# 본문 수집 설정 (7~9단계)
# ============================================================
# 통과 피드마다 테스트 기간 기사 중 이 수만큼 무작위 표본 (기사가 적으면 전부)
BODY_SAMPLE_PER_FEED = 40
BODY_SAMPLE_SEED = 42

# 기사 페이지는 피드보다 요청 간격을 넉넉히
BODY_FETCH_DELAY = 1.0
BODY_FETCH_RETRIES = 1

# 같은 사이트에서 연속으로 이 횟수만큼 차단(401/403/429)·연결 오류가 나면 그 사이트 나머지 기사는 요청하지 않음
BODY_HOST_FAIL_LIMIT = 3

HTML_HEADERS = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

MIN_BODY_WORDS = 80         # 본문 최소 단어 수 (짧은 단신도 통과하도록 낮게)
MIN_LD_COVERAGE = 0.6       # JSON-LD articleBody 대비 추출 본문 비율 하한
SHINGLE_SIZE = 5            # 두 본문을 비교하는 단위 (연속 5단어 묶음)
MIN_REF_COVERAGE = 0.8      # 기준 본문(JSON-LD 또는 사이트별 본문 영역)의 80% 미만만 trafilatura에 있으면 기준 본문 사용
RSS_FULL_COVERAGE = 0.9     # 원문 본문의 90% 이상이 RSS 본문에 있으면 RSS 본문 = 전문

# 사이트별 본문 영역 (태그, class 이름). trafilatura 결과를 교차 확인하는 기준 본문으로 사용
#   2026-11-05 각 사이트 기사 2건씩 HTML을 열어 <p>가 모여 있는 요소를 확인해서 정함
#   (rss_news 본문 시험 코드의 매체별 SELECTORS와 같은 방식)
#   등록된 사이트는 이 영역을 최종 본문으로 쓰고, trafilatura 결과와 겹침 비율로 서로 확인
SITE_BODY_CONTAINERS = {
    "motherjones.com": ("article", "entry-content"),
    "techcrunch.com": ("div", "entry-content"),
    "semiengineering.com": ("div", "post_cnt"),
    "nypost.com": ("div", "single__content"),
    "arstechnica.com": ("div", "post-content"),   # 광고를 사이에 두고 여러 개로 나뉨 → trafilatura가 첫 덩어리만 가져감
    "9to5mac.com": ("div", "post-content"),
    "wired.com": ("div", "body__inner-container"),
}

# 본문 영역 안에서 빼는 요소 (class 단어가 이 접두어로 시작)
JUNK_CLASS_PREFIXES = (
    "ad-", "ads", "advert", "newsletter", "related", "promo", "recirc", "share", "social", "subscribe",
    "inline-module",   # NY Post 본문 중간 뉴스레터 가입·관련 기사 박스
)

# 본문 영역 문단 중 링크 글자가 이 비율 이상이면 관련 기사·상품 링크 목록으로 보고 제외 (9to5Mac 쇼핑 목록 등)
MAX_LINK_DENSITY = 0.8

MIN_BODY_ARTICLES = 8       # 판정에 필요한 최소 기사 페이지 수
MIN_RSS_COMPARABLE = 5      # RSS 전문 여부 판정에 필요한 최소 비교 가능 기사 수
BODY_USABLE_RATE = 0.8
BODY_UNSTABLE_RATE = 0.5

HTML_DIR = CACHE_DIR / "html"
HTML_FETCH_LOG_PATH = CACHE_DIR / "html_fetch.jsonl"

BODY_CHECK_PATH = BASE_DIR / "body_check.parquet"
BODY_SUMMARY_PATH = BASE_DIR / "body_summary.csv"
BODY_REVIEW_PATH = BASE_DIR / "body_review.csv"
BODY_LISTS_PATH = BASE_DIR / "body_lists.json"

In [5]:
# ============================================================
# 페이지 판정 패턴 (1_google_rss_source_test.ipynb와 같은 기준)
# ============================================================
VIDEO_PATH_PATTERN = re.compile(
    r"/(video|videos|watch|clip|clips|v|live-tv|livetv|podcast|podcasts|audio|gallery|galleries|slideshow|photos)(/|$)",
    re.IGNORECASE,
)
VIDEO_HOST_PATTERN = re.compile(r"^(video|videos|watch|tv)\.", re.IGNORECASE)
LIVE_BLOG_PATH_PATTERN = re.compile(r"/(live|live-news|live-updates|liveblog)(/|$)|live-updates", re.IGNORECASE)

ARTICLE_LD_TYPES = {
    "Article", "NewsArticle", "ReportageNewsArticle", "AnalysisNewsArticle", "OpinionNewsArticle",
    "BackgroundNewsArticle", "ReviewNewsArticle", "BlogPosting", "Report",
    "Review",   # WIRED 제품 리뷰 (og:type은 website)
}

PAYWALL_PHRASES = [
    "subscribe to continue", "subscribe to read", "subscription required", "to continue reading",
    "already a subscriber", "sign in to continue", "sign in to read", "log in to continue",
    "create a free account to continue", "this article is for subscribers", "this content is for subscribers",
    "unlock this article", "become a member to read",
]

# 봇 차단 / 챌린지 페이지 흔적 (<title> 또는 HTML 원문)
# 뒤의 세 문구는 rss_news 본문 시험에서 Zacks·MarketWatch·WSJ 차단 페이지로 확인된 문구
BLOCK_TITLE_PHRASES = [
    "access denied", "just a moment", "attention required", "are you a robot", "pardon our interruption",
    "request unsuccessful", "verify you are human", "403 forbidden",
]
BLOCK_HTML_MARKERS = [
    "px-captcha", "captcha-delivery.com", "cf-chl-", "_incapsula_resource",
    "made us think you were a bot", "please enable js and disable any ad blocker",
]

PUBLISHED_DATE_FIELDS = [
    "ld:datePublished", "property:article:published_time", "itemprop:datePublished",
    "name:parsely-pub-date", "name:pubdate", "name:publish-date", "name:sailthru.date",
]

# 언론사별 제외 구역: "호스트/경로" 문자열(www. 제거)에 정규식 검색 (1_google_rss_source_test.ipynb의 SECTION_EXCLUDE_RULES)
# URL만으로 판단하므로 다운로드 전에 거름
SECTION_EXCLUDE_RULES = {
    "nypost.com": [
        r"/betting/",     # 도박 프로모션 (테스트 기간 NY Post 505건 중 25건)
    ],
}

# 아래 판정은 URL·메타데이터로 걸러낼 수 있는 기사 아닌 페이지 → 성공률 분모에서 제외
FILTERABLE_STATUSES = {"EXCLUDED_SECTION", "VIDEO_PAGE", "LIVE_BLOG", "NOT_ARTICLE"}

# ============================================================
# RSS 본문 처리 패턴
# ============================================================
# RSS 본문 끝에 붙는 출처 문구 (기사 본문이 아니므로 제거)
#   WordPress 기본 "The post ... appeared first on ...", Fortune "This story was originally featured on ..." (rss_news 감사에서 확인)
RSS_FOOTER_PATTERN = re.compile(
    r"^(the post .+ appeared first on .+|this (story|article) (was )?originally (appeared|featured|published) .+)$",
    re.IGNORECASE,
)

# RSS 본문이 티저(앞부분만)로 끝나는 흔적. rss_news 감사에서 The Motley Fool이 "Continue reading"으로 끝나는 사례 확인
RSS_TEASER_PATTERN = re.compile(r"(continue reading|read more|keep reading|\[…\]|\[\.\.\.\]|…|\.\.\.)\W*$", re.IGNORECASE)

# 추출한 본문에 섞이면 안 되는 문구 (줄 단위 검사, 검증용)
#   rss_news 매체 검토에서 정제 필요로 기록된 유형: 뉴스레터·앱 홍보, 추천/관련 기사, 출처 푸터
BODY_JUNK_PATTERNS = {
    "newsletter": re.compile(r"sign up for .{0,40}newsletter|subscribe to (our|the) .{0,30}newsletter", re.IGNORECASE),
    "app_promo": re.compile(r"download the .{0,30}app", re.IGNORECASE),
    "related": re.compile(r"^(read more|related|recommended|more from|see also|most popular|trending)\b.{0,60}$", re.IGNORECASE),
    "advertisement": re.compile(r"^advertisement$", re.IGNORECASE),
    "rss_footer": re.compile(r"appeared first on|originally (appeared|featured) on", re.IGNORECASE),
}

## 0-1. 공통 유틸 (요청, 캐시, 피드 파싱)

In [6]:
FEED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 캐시 폴더는 git에 올리지 않음
(CACHE_DIR / ".gitignore").write_text("*\n")


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path):
    path = Path(path)

    if not path.exists():
        return []

    records = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def now_utc():
    return datetime.now(timezone.utc)


def url_hash(url):
    return hashlib.sha1(url.encode("utf-8")).hexdigest()


def et_day_bounds(start, end):
    # 동부시간 날짜 구간 [start, end] → UTC [시작, 끝)
    start_dt = datetime.combine(start, datetime.min.time(), NEWS_TIMEZONE)
    end_dt = datetime.combine(end + timedelta(days=1), datetime.min.time(), NEWS_TIMEZONE)

    return start_dt.astimezone(timezone.utc), end_dt.astimezone(timezone.utc)


def date_range(start, end):
    days = []
    current = start

    while current <= end:
        days.append(current)
        current += timedelta(days=1)

    return days


def to_feed_dict(feed_ids, extra=None):
    # feed_id 리스트 → {feed_id: 후보 정보 (+추가 정보)} (CANDIDATE_FEEDS 순서 유지)
    feed_ids = set(feed_ids)
    extra = extra or {}

    return {
        feed_id: {**info, **extra.get(feed_id, {})}
        for feed_id, info in CANDIDATE_FEEDS.items()
        if feed_id in feed_ids
    }


def make_session():
    session = requests.Session()
    session.headers.update(REQUEST_HEADERS)

    return session


SESSION = make_session()
_last_request_at = {}


def polite_get(url, delay=REQUEST_DELAY, headers=None, retries=MAX_RETRIES):
    # 같은 호스트에 delay초 간격을 두고 요청, 429/5xx/네트워크 오류는 retries번까지 재시도
    # headers: 기본 헤더(RSS용)에 덮어쓸 헤더 (기사 HTML 요청은 HTML_HEADERS)
    host = urlparse(url).hostname

    for attempt in range(retries + 1):
        wait = _last_request_at.get(host, 0) + delay - time.time()

        if wait > 0:
            time.sleep(wait)

        try:
            # archive.org는 Accept에 RSS 형식을 지정하면 요청 제한과 상관없이 바로 429를 돌려줌
            if host == "web.archive.org":
                headers = {"Accept": "*/*"}

            response = SESSION.get(url, timeout=REQUEST_TIMEOUT, headers=headers)
            error = None

        except requests.RequestException as e:
            response = None
            error = f"{type(e).__name__}: {e}"

        finally:
            _last_request_at[host] = time.time()

        retryable = response is None or response.status_code in {429, 500, 502, 503, 504}

        if not retryable or attempt == retries:
            return response, error

        if host == "web.archive.org":
            time.sleep(WAYBACK_BLOCK_WAIT * (attempt + 1))
        else:
            time.sleep(5 * (attempt + 1))


def entry_published(entry):
    # feedparser가 UTC로 정규화한 시각 (published → updated 순)
    for key in ("published_parsed", "updated_parsed"):
        value = entry.get(key)

        if value:
            published = datetime.fromtimestamp(calendar.timegm(value), timezone.utc)
            return published if published >= MIN_VALID_DATE else None

    return None


def parse_feed(content):
    parsed = feedparser.parse(content)
    entries = []

    for entry in parsed.entries:
        published_utc = entry_published(entry)

        entries.append({
            # Atom 피드(The Hill 등)는 제목에 &#8216; 같은 엔티티가 남아 있어 한 번 더 해제
            "title": html.unescape(entry.get("title") or "").strip(),
            "link": entry.get("link"),
            "guid": entry.get("id") or entry.get("link"),
            "published_utc": published_utc,
            "published_et": published_utc.astimezone(NEWS_TIMEZONE) if published_utc else None,
            "summary": html.unescape(entry.get("summary") or "").strip(),
            # RSS 본문 HTML (content:encoded / Atom content). 제공하지 않는 피드는 빈 문자열
            "content": "\n".join(c.get("value") or "" for c in entry.get("content", [])),
            "author": entry.get("author"),
            "tags": [t.get("term") for t in entry.get("tags", []) if t.get("term")],
        })

    return entries


def fetch_feed(url, use_cache=False, delay=REQUEST_DELAY):
    '''
    피드 요청 → {"url", "http_status", "error", "entries", "from_cache"}
    use_cache=True면 200/404 응답을 저장해두고 재요청하지 않음 (과거 날짜 아카이브 전용)
    '''
    path_ok = FEED_CACHE_DIR / f"{url_hash(url)}.xml.gz"
    path_404 = FEED_CACHE_DIR / f"{url_hash(url)}.404"

    if use_cache and path_ok.exists():
        content = gzip.decompress(path_ok.read_bytes())
        return {"url": url, "http_status": 200, "error": None, "entries": parse_feed(content), "from_cache": True}

    if use_cache and path_404.exists():
        return {"url": url, "http_status": 404, "error": None, "entries": [], "from_cache": True}

    response, error = polite_get(url, delay=delay)

    if response is None:
        return {"url": url, "http_status": None, "error": error, "entries": [], "from_cache": False}

    entries = parse_feed(response.content) if response.status_code == 200 else []

    if use_cache and response.status_code == 200:
        path_ok.write_bytes(gzip.compress(response.content))

    if use_cache and response.status_code == 404:
        path_404.touch()

    return {"url": url, "http_status": response.status_code, "error": None, "entries": entries, "from_cache": False}


def add_param(url, key, value):
    return f"{url}{'&' if '?' in url else '?'}{key}={value}"


def entry_times(entries):
    return [e["published_utc"] for e in entries if e["published_utc"]]

## 1. 라이브 피드 점검

현재 피드가 응답하는지, 기사에 날짜가 있는지, 갱신이 멈추지 않았는지 확인합니다.
`span_hours`는 피드 하나가 담고 있는 기간입니다 (최신 기사 ~ 가장 오래된 기사).

In [7]:
def check_live(feeds):
    rows = []

    for feed_id, info in feeds.items():
        result = fetch_feed(info["url"])
        entries = result["entries"]
        times = entry_times(entries)

        if result["http_status"] != 200:
            status = "HTTP_ERROR"
        elif not entries:
            status = "EMPTY"
        elif len(times) / len(entries) < MIN_DATE_RATE:
            status = "NO_DATE"
        elif max(times) < now_utc() - timedelta(days=STALE_DAYS):
            status = "STALE"
        else:
            status = "OK"

        rows.append({
            "feed_id": feed_id,
            "source": info["source"],
            "name": info["name"],
            "section": info["section"],
            "live_status": status,
            "http_status": result["http_status"],
            "entries": len(entries),
            "date_rate": round(len(times) / len(entries), 2) if entries else None,
            "newest_et": max(times).astimezone(NEWS_TIMEZONE) if times else None,
            "oldest_et": min(times).astimezone(NEWS_TIMEZONE) if times else None,
            "span_hours": round((max(times) - min(times)).total_seconds() / 3600, 1) if times else None,
            "error": result["error"],
        })

    return pd.DataFrame(rows)


live_df = check_live(CANDIDATE_FEEDS)
live_df.to_csv(LIVE_RESULT_PATH, index=False, encoding="utf-8-sig")

# ── 변수 정리: 라이브 점검 ──
LIVE_OK_FEEDS = to_feed_dict(live_df.loc[live_df["live_status"] == "OK", "feed_id"])
LIVE_FAILED_FEEDS = (
    live_df.loc[live_df["live_status"] != "OK"]
    .set_index("feed_id")["live_status"]
    .to_dict()
)

print(f"라이브 정상 {len(LIVE_OK_FEEDS)}개 / 실패 {len(LIVE_FAILED_FEEDS)}개")
print("실패:", LIVE_FAILED_FEEDS)

live_df[["feed_id", "name", "section", "live_status", "http_status", "entries", "date_rate", "newest_et", "span_hours"]]

라이브 정상 72개 / 실패 9개
실패: {'marketwatch_pulse': 'STALE', 'barrons': 'HTTP_ERROR', 'yahoo_finance': 'HTTP_ERROR', 'zacks': 'HTTP_ERROR', 'usatoday_money': 'HTTP_ERROR', 'cnn_money': 'STALE', 'wapo_business': 'HTTP_ERROR', 'apnews': 'HTTP_ERROR', 'globenewswire': 'HTTP_ERROR'}


,feed_id,name,section,live_status,http_status,entries,date_rate,newest_et,span_hours
0,cnbc_top,CNBC,Top News,OK,200,30,1.0,2026-11-04 21:07:35-05:00,14.0
1,cnbc_business,CNBC,Business,OK,200,30,1.0,2026-11-04 16:47:22-05:00,176.5
2,cnbc_investing,CNBC,Investing,OK,200,30,1.0,2026-11-04 19:42:58-05:00,127.3
3,cnbc_market_insider,CNBC,Market Insider,OK,200,30,1.0,2026-11-04 17:04:55-05:00,341.3
4,cnbc_economy,CNBC,Economy,OK,200,30,1.0,2026-11-04 17:59:27-05:00,489.3
...,...,...,...,...,...,...,...,...,...
76,semiengineering,Semiconductor Engineering,All,OK,200,10,1.0,2026-11-04 03:15:25-05:00,14.2
77,venturebeat,VentureBeat,All,OK,200,7,1.0,2026-11-04 19:55:41-05:00,76.9
78,businesswire,Business Wire,All,OK,200,23,1.0,2026-11-04 21:00:00-05:00,14.2
79,prnewswire,PR Newswire,All,OK,200,20,1.0,2026-11-04 21:12:00-05:00,1.9


## 2. 과거 접근 방식 판별 + 깊이 테스트

### 2-1. `DATE_ARCHIVE`: WordPress 날짜 아카이브 피드

사이트 전체 피드(`/feed/`)인 후보만 대상입니다. RSS(`/feed/`)가 안 되면 Atom(`/feed/atom/`)도 시도합니다.
(The Hill은 날짜 아카이브 RSS의 1페이지만 비어 있고 Atom은 정상인 사례)

In [8]:
ARCHIVE_FORMATS = {
    "rss": "{root}/{day:%Y/%m/%d}/feed/",
    "atom": "{root}/{day:%Y/%m/%d}/feed/atom/",
}


def site_root(url):
    parsed = urlparse(url)

    return f"{parsed.scheme}://{parsed.netloc}"


def is_site_wide_wp_feed(url):
    return urlparse(url).path.rstrip("/") == "/feed"


def archive_url(info, day, page=1, fmt="rss"):
    url = ARCHIVE_FORMATS[fmt].format(root=site_root(info["url"]), day=day)

    return url if page == 1 else add_param(url, "paged", page)


def archive_match_rate(entries, day):
    # 아카이브 기사 중 요청 날짜(동부시간 ±1일)에 속하는 비율, 정확히 그 날짜인 비율
    days = [t.astimezone(NEWS_TIMEZONE).date() for t in entry_times(entries)]

    if not days:
        return 0.0, 0.0

    near = sum(abs((d - day).days) <= 1 for d in days) / len(days)
    exact = sum(d == day for d in days) / len(days)

    return near, exact


def check_date_archive(feeds, test_day, depth_days):
    rows = []

    for feed_id, info in feeds.items():
        if not is_site_wide_wp_feed(info["url"]):
            continue

        row = {"feed_id": feed_id, "archive_ok": False, "archive_format": None}

        for fmt in ARCHIVE_FORMATS:
            result = fetch_feed(archive_url(info, test_day, fmt=fmt), use_cache=True)
            near, exact = archive_match_rate(result["entries"], test_day)

            row.update({
                "archive_http": result["http_status"],
                "archive_entries": len(result["entries"]),
                "archive_near_rate": round(near, 2),
                "archive_exact_rate": round(exact, 2),
            })

            if result["entries"] and near >= MIN_ARCHIVE_MATCH_RATE:
                row.update({"archive_ok": True, "archive_format": fmt})
                break

        # 깊이 테스트: 과거 날짜의 아카이브 1페이지가 나오는지
        if row["archive_ok"]:
            for day in depth_days:
                result = fetch_feed(archive_url(info, day, fmt=row["archive_format"]), use_cache=True)
                near, _ = archive_match_rate(result["entries"], day)
                row[f"depth_{day}"] = bool(result["entries"]) and near >= MIN_ARCHIVE_MATCH_RATE

        rows.append(row)

    return pd.DataFrame(rows)


archive_df = check_date_archive(CANDIDATE_FEEDS, TEST_START, DEPTH_PROBE_DATES)
archive_df

,feed_id,archive_ok,archive_format,archive_http,archive_entries,archive_near_rate,archive_exact_rate,depth_2025-06-10,depth_2024-10-15,depth_2023-10-17,depth_2022-10-18
0,barrons,False,None,403.0,0,0.0,0.0,NaN,NaN,NaN,NaN
1,ibd,True,rss,200.0,63,1.0,1.0,True,True,True,True
2,marketbeat,False,None,404.0,0,0.0,0.0,NaN,NaN,NaN,NaN
3,fortune,False,None,404.0,0,0.0,0.0,NaN,NaN,NaN,NaN
4,nypost,True,rss,200.0,20,1.0,1.0,True,True,True,True
5,axios,False,None,404.0,0,0.0,0.0,NaN,NaN,NaN,NaN
6,thehill,True,atom,200.0,15,1.0,1.0,True,True,True,True
7,time,False,None,406.0,0,0.0,0.0,NaN,NaN,NaN,NaN
8,washington_examiner,False,None,404.0,0,0.0,0.0,NaN,NaN,NaN,NaN
9,nationalreview,True,rss,200.0,19,1.0,1.0,True,True,True,True


### 2-2. `PAGED`: 최신순 피드 페이지 이동

날짜 아카이브가 안 되는 라이브 정상 피드만 대상입니다.
`paged=2` → `page=2` 순서로 시도해서 2페이지가 1페이지와 다르고 더 과거 기사면 통과,
이후 페이지 번호를 2배씩 늘리고 이진탐색으로 **마지막 페이지와 가장 오래된 기사 날짜**를 찾습니다.

In [9]:
PAGE_PARAMS = ["paged", "page"]


def paged_url(info, param, page):
    return info["url"] if page == 1 else add_param(info["url"], param, page)


def page_oldest(info, param, page, memo):
    # 페이지의 가장 오래된 기사 시각 (빈 페이지/오류면 None)
    if page not in memo:
        result = fetch_feed(paged_url(info, param, page))
        times = entry_times(result["entries"])
        memo[page] = (min(times), max(times), result["entries"]) if times else None

    return memo[page]


def find_last_page(info, param, memo):
    '''
    1, 2, 4, 8 … 로 비는 페이지를 찾고 그 사이를 이진탐색.
    파라미터를 무시하고 같은 내용을 주는 사이트가 있어서
    앞 페이지보다 더 과거 기사가 있어야 유효한 페이지로 봄
    '''
    def is_deeper(page, than):
        current = page_oldest(info, param, page, memo)
        return current is not None and current[0] < page_oldest(info, param, than, memo)[0]

    good, bad = 1, None
    page = 2

    while page <= MAX_PAGE_NUMBER:
        if is_deeper(page, good):
            good = page
            page *= 2
        else:
            bad = page
            break

    if bad is None:
        return good

    while bad - good > 1:
        mid = (good + bad) // 2

        if is_deeper(mid, good):
            good = mid
        else:
            bad = mid

    return good


def check_paged(feeds):
    rows = []

    for feed_id, info in feeds.items():
        row = {"feed_id": feed_id, "paged_ok": False, "page_param": None}
        first = fetch_feed(info["url"])
        first_links = {e["link"] for e in first["entries"]}
        first_times = entry_times(first["entries"])

        for param in PAGE_PARAMS:
            memo = {}
            second = page_oldest(info, param, 2, memo)

            if not second or not first_times:
                continue

            second_links = {e["link"] for e in second[2]}

            # 2페이지가 1페이지와 다르고, 2페이지의 최신 기사가 1페이지 중간보다 과거여야 함
            is_older = second[1] <= sorted(first_times)[len(first_times) // 2]

            if second_links != first_links and is_older:
                last_page = find_last_page(info, param, memo)
                oldest = page_oldest(info, param, last_page, memo)[0]

                row.update({
                    "paged_ok": True,
                    "page_param": param,
                    "page_size": len(first["entries"]),
                    "last_page": last_page,
                    "paged_oldest_et": oldest.astimezone(NEWS_TIMEZONE),
                })
                break

        rows.append(row)

    return pd.DataFrame(rows)


archive_ok_ids = set(archive_df.loc[archive_df["archive_ok"], "feed_id"])

paged_df = check_paged({
    feed_id: info
    for feed_id, info in LIVE_OK_FEEDS.items()
    if feed_id not in archive_ok_ids
})

paged_df[paged_df["paged_ok"]]

,feed_id,paged_ok,page_param,page_size,last_page,paged_oldest_et
0,cnbc_top,True,page,30.0,7.0,2026-10-29 15:36:04-04:00
1,cnbc_business,True,page,30.0,7.0,2026-09-11 11:36:45-04:00
2,cnbc_investing,True,page,30.0,7.0,2026-10-07 15:23:51-04:00
3,cnbc_market_insider,True,page,30.0,7.0,2026-07-10 13:53:52-04:00
4,cnbc_economy,True,page,30.0,7.0,2026-05-21 04:42:03-04:00
5,cnbc_earnings,True,page,30.0,7.0,2026-08-06 17:24:38-04:00
6,cnbc_finance,True,page,30.0,7.0,2026-09-04 15:07:16-04:00
7,cnbc_tech,True,page,30.0,7.0,2026-10-16 08:38:58-04:00
21,nasdaq_markets,True,paged,15.0,2.0,2026-11-04 19:33:59-05:00
33,nypost_business,True,paged,20.0,10.0,2026-10-21 14:06:10-04:00


### 2-3. (참고용) archive.org 피드 스냅샷 유무

**수집에는 쓰지 않습니다** (맨 위 '수집에서 제외한 방식' 참고). 과거 접근 방법이 없는 피드 중
Wayback 스냅샷이라도 남아 있는 피드를 기록해두기 위한 조회입니다.
CDX API로 테스트 기간(앞뒤 하루 포함)의 스냅샷 수와 최대 간격을 보고, 스냅샷이 있으면 깊이 테스트 날짜별 스냅샷 수도 셉니다.

In [10]:
CDX_URL = "https://web.archive.org/cdx/search/cdx"


def wayback_snapshots(feed_url, start_utc, end_utc):
    # [start_utc, end_utc] 사이 스냅샷 [(capture_utc, original_url)] (과거 구간만 캐시)
    key = f"{feed_url}|{start_utc:%Y%m%d%H%M%S}|{end_utc:%Y%m%d%H%M%S}"

    for record in load_jsonl(CDX_CACHE_PATH):
        if record["key"] == key:
            return [(datetime.fromisoformat(t), u) for t, u in record["snapshots"]]

    params = (
        f"?url={requests.utils.quote(feed_url, safe='')}"
        f"&from={start_utc:%Y%m%d%H%M%S}&to={end_utc:%Y%m%d%H%M%S}"
        "&output=json&filter=statuscode:200&collapse=digest&fl=timestamp,original"
    )
    response, error = polite_get(CDX_URL + params, delay=WAYBACK_DELAY)

    if response is None or response.status_code != 200:
        raise RuntimeError(f"CDX 실패: {feed_url} {error or response.status_code}")

    try:
        rows = response.json()[1:] if response.text.strip() else []
    except ValueError:
        raise RuntimeError(f"CDX 응답 해석 실패: {feed_url}")
    snapshots = [
        (datetime.strptime(ts, "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc), original)
        for ts, original in rows
    ]

    if end_utc < now_utc() - timedelta(days=2):
        append_jsonl(CDX_CACHE_PATH, {
            "key": key,
            "snapshots": [(t.isoformat(), u) for t, u in snapshots],
        })

    return snapshots


def check_wayback(feeds, start, end, depth_days):
    window_start, window_end = et_day_bounds(start, end)
    rows = []

    for feed_id, info in feeds.items():
        row = {"feed_id": feed_id}

        try:
            snapshots = wayback_snapshots(
                info["url"],
                window_start - timedelta(days=1),
                window_end + timedelta(days=1),
            )

        except RuntimeError as e:
            rows.append({**row, "wayback_error": str(e)})
            continue

        in_window = [t for t, _ in snapshots if window_start <= t <= window_end]
        edges = [window_start] + in_window + [window_end]
        gaps = [(b - a).total_seconds() / 3600 for a, b in zip(edges, edges[1:])]

        row.update({
            "wayback_snapshots": len(in_window),
            "wayback_max_gap_hours": round(max(gaps), 1),
        })

        if in_window:
            for day in depth_days:
                day_start, day_end = et_day_bounds(day, day)
                try:
                    row[f"wb_depth_{day}"] = len(wayback_snapshots(info["url"], day_start, day_end))
                except RuntimeError:
                    row[f"wb_depth_{day}"] = None

        rows.append(row)

    return pd.DataFrame(rows)


wayback_df = check_wayback(CANDIDATE_FEEDS, TEST_START, TEST_END, DEPTH_PROBE_DATES)
wayback_df[wayback_df["wayback_snapshots"].fillna(0) > 0]

,feed_id,wayback_snapshots,wayback_max_gap_hours,wb_depth_2025-06-10,wb_depth_2024-10-15,wb_depth_2023-10-17,wb_depth_2022-10-18,wayback_error
0,cnbc_top,132.0,2.3,0.0,0.0,0.0,1.0,NaN
8,wsj_markets,190.0,1.6,0.0,0.0,0.0,0.0,NaN
9,wsj_business,186.0,1.0,0.0,0.0,0.0,0.0,NaN
10,wsj_economy,189.0,0.8,0.0,0.0,0.0,0.0,NaN
11,wsj_tech,188.0,1.6,0.0,0.0,0.0,0.0,NaN
12,wsj_world,189.0,0.9,0.0,0.0,0.0,0.0,NaN
13,marketwatch_top,193.0,2.4,0.0,0.0,0.0,0.0,NaN
18,bloomberg_economics,183.0,1.0,0.0,0.0,0.0,0.0,NaN
21,seekingalpha_all,152.0,1.1,0.0,0.0,0.0,0.0,NaN
25,investing_news,154.0,1.2,0.0,0.0,0.0,0.0,NaN


### 2-4. 피드별 수집 방식 결정

우선순위: `DATE_ARCHIVE` → `PAGED`(테스트 시작일까지 내려가는 경우).
둘 다 안 되지만 테스트 기간 Wayback 스냅샷은 있는 피드는 `WAYBACK_ONLY_FEEDS`로 따로 기록합니다.

In [11]:
access_df = (
    live_df[["feed_id", "source", "name", "section", "live_status", "entries", "span_hours"]]
    .merge(archive_df, on="feed_id", how="left")
    .merge(paged_df, on="feed_id", how="left")
    .merge(wayback_df, on="feed_id", how="left")
)

test_start_utc, _ = et_day_bounds(TEST_START, TEST_END)

access_df["paged_reaches_test"] = (
    access_df["paged_oldest_et"].notna()
    & (pd.to_datetime(access_df["paged_oldest_et"], utc=True) <= test_start_utc)
)


def choose_method(row):
    if row["archive_ok"] == True:
        return "DATE_ARCHIVE"

    if row["paged_reaches_test"]:
        return "PAGED"

    return None


access_df["method"] = access_df.apply(choose_method, axis=1)
access_df.to_csv(ACCESS_RESULT_PATH, index=False, encoding="utf-8-sig")

# ── 변수 정리: 접근 방식 ──
DATE_ARCHIVE_FEEDS = to_feed_dict(access_df.loc[access_df["method"] == "DATE_ARCHIVE", "feed_id"])
PAGED_FEEDS = to_feed_dict(access_df.loc[access_df["method"] == "PAGED", "feed_id"])
NO_PAST_ACCESS_FEEDS = to_feed_dict(access_df.loc[access_df["method"].isna(), "feed_id"])

# 과거 접근은 안 되지만 테스트 기간 Wayback 스냅샷은 있는 피드 (참고용, 수집 안 함)
WAYBACK_ONLY_FEEDS = to_feed_dict(
    access_df.loc[access_df["method"].isna() & (access_df["wayback_snapshots"].fillna(0) > 0), "feed_id"]
)

for label, feeds in [
    ("DATE_ARCHIVE", DATE_ARCHIVE_FEEDS),
    ("PAGED", PAGED_FEEDS),
    ("과거 접근 불가", NO_PAST_ACCESS_FEEDS),
    ("└ Wayback만 있음", WAYBACK_ONLY_FEEDS),
]:
    print(f"{label:14s} {len(feeds):2d}개: {list(feeds)}")

depth_cols = [c for c in access_df.columns if c.startswith("depth_")]

access_df.loc[
    access_df["method"].notna(),
    ["feed_id", "name", "section", "method", "archive_format", "page_param", "page_size",
     "last_page", "paged_oldest_et"] + depth_cols,
]

DATE_ARCHIVE    9개: ['ibd', 'nypost', 'thehill', 'nationalreview', 'motherjones', 'techcrunch', 'arstechnica', '9to5mac', 'semiengineering']
PAGED           3개: ['fastcompany', 'wired', 'eetimes']
과거 접근 불가       69개: ['cnbc_top', 'cnbc_business', 'cnbc_investing', 'cnbc_market_insider', 'cnbc_economy', 'cnbc_earnings', 'cnbc_finance', 'cnbc_tech', 'wsj_markets', 'wsj_business', 'wsj_economy', 'wsj_tech', 'wsj_world', 'marketwatch_top', 'marketwatch_bulletins', 'marketwatch_pulse', 'barrons', 'bloomberg_markets', 'bloomberg_economics', 'bloomberg_tech', 'seekingalpha_all', 'seekingalpha_currents', 'fool', 'nasdaq_markets', 'investing_news', 'investing_economy', 'marketbeat', 'thestreet', 'kiplinger', 'yahoo_finance', 'zacks', 'fortune', 'business_insider', 'foxbusiness_latest', 'foxbusiness_markets', 'foxbusiness_economy', 'cbs_moneywatch', 'nypost_business', 'usatoday_money', 'cnn_money', 'forbes_business', 'forbes_innovation', 'qz', 'axios', 'semafor', 'nyt_business', 'nyt_economy', '

,feed_id,name,section,method,archive_format,page_param,page_size,last_page,paged_oldest_et,depth_2025-06-10,depth_2024-10-15,depth_2023-10-17,depth_2022-10-18
20,ibd,Investor's Business Daily,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
38,nypost,New York Post,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
44,fastcompany,Fast Company,Latest,PAGED,NaN,page,20.0,10532.0,1995-01-01 05:00:00-05:00,NaN,NaN,NaN,NaN
62,thehill,The Hill,All,DATE_ARCHIVE,atom,NaN,NaN,NaN,NaT,True,True,True,True
65,nationalreview,National Review,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
66,motherjones,Mother Jones,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
67,techcrunch,TechCrunch,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
69,arstechnica,Ars Technica,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
70,wired,WIRED,All,PAGED,NaN,page,50.0,200.0,2024-06-07 13:52:21-04:00,NaN,NaN,NaN,NaN
73,9to5mac,9to5Mac,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True


## 3. 테스트 기간 실제 수집

2단계에서 정한 방식으로 테스트 기간의 기사를 실제로 모읍니다. 두 수집 함수는 5단계 실사용 함수에서도 그대로 씁니다.

- `DATE_ARCHIVE`: 사이트 시간대가 동부시간이 아닐 수 있어 앞뒤 하루 아카이브까지 받은 뒤 동부시간 날짜로 거름.
  날짜마다 빈 페이지(404)가 나올 때까지 받았으면 `complete=True`
- `PAGED`: 기간 끝이 들어있는 페이지를 이진탐색으로 찾고 기간 시작보다 과거가 나올 때까지 다음 페이지로 진행.
  기간 시작보다 과거 기사가 나오기 전에 페이지가 끝나면 `complete=False`

In [12]:
def finalize_articles(feed_id, entries, start, end, method):
    # 기간 필터 + 링크 기준 중복 제거
    window_start, window_end = et_day_bounds(start, end)
    info = CANDIDATE_FEEDS[feed_id]
    seen = {}

    for entry in entries:
        published = entry["published_utc"]

        if published is None or not (window_start <= published < window_end):
            continue

        key = entry["link"] or entry["guid"]

        if key not in seen or published < seen[key]["published_utc"]:
            seen[key] = entry

    return [
        {
            "feed_id": feed_id,
            "source": info["source"],
            "name": info["name"],
            "section": info["section"],
            "method": method,
            **entry,
        }
        for entry in seen.values()
    ]


def collect_date_archive(feed_id, info, start, end, fmt="rss"):
    entries = []
    complete = True
    requests_made = 0

    for day in date_range(start - timedelta(days=1), end + timedelta(days=1)):
        seen_links = set()

        for page in range(1, MAX_PAGES_PER_DAY + 1):
            result = fetch_feed(archive_url(info, day, page, fmt), use_cache=day < date.today() - timedelta(days=1))
            requests_made += 1
            links = {e["link"] for e in result["entries"]}

            # 마지막 페이지 다음은 404 또는 빈 피드, 같은 페이지 반복도 종료로 봄
            if not result["entries"] or links <= seen_links:
                if result["http_status"] not in {200, 404}:
                    complete = False
                break

            seen_links |= links
            entries.extend(result["entries"])

        else:
            complete = False

    return entries, {"complete": complete, "requests": requests_made}


def collect_paged(feed_id, info, start, end, param="paged"):
    window_start, window_end = et_day_bounds(start, end)
    memo = {}

    # 기간 끝보다 과거 기사가 처음 나오는 페이지를 찾음 (2배씩 늘린 뒤 이진탐색)
    def reaches(page):
        oldest = page_oldest(info, param, page, memo)
        return oldest is None or oldest[0] < window_end

    lo, hi = 1, 1

    while not reaches(hi):
        lo, hi = hi, min(hi * 2, MAX_PAGE_NUMBER)

        if hi == lo:
            return [], {"complete": False, "requests": len(memo)}

    while hi - lo > 1:
        mid = (lo + hi) // 2

        if reaches(mid):
            hi = mid
        else:
            lo = mid

    # 페이지 밀림 대비 한 페이지 앞부터 기간 시작보다 과거가 나올 때까지
    entries = []
    page = max(1, hi - 1)
    complete = False

    while page <= MAX_PAGE_NUMBER:
        result = page_oldest(info, param, page, memo)

        if result is None:
            break

        entries.extend(result[2])

        if result[0] < window_start:
            complete = True
            break

        page += 1

    return entries, {"complete": complete, "requests": len(memo)}


def collect_feed(feed_id, info, start, end):
    method = info["method"]

    if method == "DATE_ARCHIVE":
        entries, stats = collect_date_archive(feed_id, info, start, end, fmt=info.get("archive_format") or "rss")
    elif method == "PAGED":
        entries, stats = collect_paged(feed_id, info, start, end, param=info.get("page_param") or "paged")
    else:
        raise ValueError(f"알 수 없는 방식: {method}")

    return finalize_articles(feed_id, entries, start, end, method), stats

In [13]:
METHOD_PARAMS = (
    access_df.loc[access_df["method"].notna()]
    .set_index("feed_id")[["method", "archive_format", "page_param"]]
    .astype(object)
    .where(lambda d: d.notna(), None)
    .to_dict(orient="index")
)

TEST_TARGET_FEEDS = to_feed_dict(METHOD_PARAMS, extra=METHOD_PARAMS)

collect_rows = []
all_articles = []

for feed_id, info in TEST_TARGET_FEEDS.items():
    started = time.time()

    try:
        articles, stats = collect_feed(feed_id, info, TEST_START, TEST_END)
        error = None

    except Exception as e:
        articles, stats = [], {"complete": False}
        error = f"{type(e).__name__}: {e}"

    all_articles.extend(articles)
    collect_rows.append({
        "feed_id": feed_id,
        "method": info["method"],
        "articles": len(articles),
        **stats,
        "seconds": round(time.time() - started, 1),
        "error": error,
    })
    print(f"{feed_id:22s} {info['method']:12s} {len(articles):5d}건  {stats}  {error or ''}", flush=True)

collect_df = pd.DataFrame(collect_rows)
articles_df = pd.DataFrame(all_articles)
articles_df.to_parquet(ARTICLES_PATH, index=False)

print(f"\n저장: {ARTICLES_PATH} ({len(articles_df)}건)")

ibd                    DATE_ARCHIVE   131건  {'complete': True, 'requests': 8}  


nypost                 DATE_ARCHIVE   505건  {'complete': True, 'requests': 56}  


fastcompany            PAGED          108건  {'complete': True, 'requests': 24}  


thehill                DATE_ARCHIVE   290건  {'complete': True, 'requests': 41}  


nationalreview         DATE_ARCHIVE    47건  {'complete': True, 'requests': 11}  


motherjones            DATE_ARCHIVE    15건  {'complete': True, 'requests': 8}  


... (3 outputs left out)

9to5mac                DATE_ARCHIVE    59건  {'complete': True, 'requests': 8}  


eetimes                PAGED            9건  {'complete': True, 'requests': 15}  


semiengineering        DATE_ARCHIVE    23건  {'complete': True, 'requests': 9}  



저장: newsrss_test/articles_2025-10-14_2025-10-15.parquet (1295건)


## 4. 판정 + 피드 딕셔너리 정리

- `PASSED`: 기사가 있고 끝까지 순회함 (날짜별 마지막 페이지까지 / 기간 시작보다 과거 페이지까지)
- `PARTIAL`: 기사는 있지만 순회가 중간에 끊김 (페이지 상한, 요청 오류)
- `LIVE_ONLY`: 지금 피드는 정상이지만 과거로 갈 방법이 없음 → 앞으로 주기적으로 받아 쌓는 용도로만 사용 가능

`WAYBACK_ONLY_FEEDS`는 판정과 별개인 참고 목록입니다. `LIVE_ONLY`·`FAILED` 중 테스트 기간 Wayback 스냅샷이 있는 피드가 들어갑니다.
- `FAILED`: 라이브도 실패하고 과거 접근도 불가, 또는 수집 결과 0건

In [14]:
# 시간대별 공백 확인용: 테스트 기간 중 기사가 0건인 시간(동부시간) 수
def empty_hours(group):
    window_start, window_end = et_day_bounds(TEST_START, TEST_END)
    hours = pd.date_range(window_start, window_end, freq="h", inclusive="left")
    filled = set(pd.to_datetime(group["published_utc"], utc=True).dt.floor("h"))

    return sum(h not in filled for h in hours)


per_day = (
    articles_df
    .assign(day=lambda d: d["published_et"].map(lambda t: t.date().isoformat()))
    .pivot_table(index="feed_id", columns="day", values="link", aggfunc="count", fill_value=0)
)

summary_df = (
    access_df[["feed_id", "source", "name", "section", "live_status", "method"]]
    .merge(collect_df.drop(columns="method"), on="feed_id", how="left")
    .merge(per_day, left_on="feed_id", right_index=True, how="left")
)

summary_df["empty_hours"] = summary_df["feed_id"].map(
    articles_df.groupby("feed_id").apply(empty_hours, include_groups=False)
)


def verdict(row):
    if row["method"] is None or pd.isna(row["method"]):
        return "LIVE_ONLY" if row["live_status"] == "OK" else "FAILED"

    if not row["articles"] or pd.isna(row["articles"]):
        return "FAILED"

    return "PASSED" if row["complete"] == True else "PARTIAL"


summary_df["verdict"] = summary_df.apply(verdict, axis=1)
summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

day_cols = list(per_day.columns)

summary_df[
    ["feed_id", "name", "section", "method", "verdict", "articles"] + day_cols
    + ["empty_hours", "complete", "requests", "seconds", "error"]
].sort_values(["verdict", "method", "articles"], ascending=[False, True, False])

,feed_id,name,section,method,verdict,articles,2025-10-14,2025-10-15,empty_hours,complete,requests,seconds,error
38,nypost,New York Post,All,DATE_ARCHIVE,PASSED,505.0,246.0,259.0,2.0,True,56.0,0.5,None
62,thehill,The Hill,All,DATE_ARCHIVE,PASSED,290.0,156.0,134.0,6.0,True,41.0,0.2,None
20,ibd,Investor's Business Daily,All,DATE_ARCHIVE,PASSED,131.0,63.0,68.0,25.0,True,8.0,0.1,None
73,9to5mac,9to5Mac,All,DATE_ARCHIVE,PASSED,59.0,27.0,32.0,19.0,True,8.0,0.1,None
67,techcrunch,TechCrunch,All,DATE_ARCHIVE,PASSED,55.0,28.0,27.0,21.0,True,11.0,0.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
40,usatoday_money,USA Today,Money,None,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,cnn_money,CNN Business,Money,None,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52,wapo_business,Washington Post,Business,None,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59,apnews,Associated Press,All,None,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# ── 변수 정리: 최종 판정 ──
def feeds_where(mask, with_method=True):
    ids = summary_df.loc[mask, "feed_id"]

    return to_feed_dict(ids, extra=METHOD_PARAMS if with_method else None)


PASSED_FEEDS = feeds_where(summary_df["verdict"] == "PASSED")
PASSED_DATE_ARCHIVE_FEEDS = {k: v for k, v in PASSED_FEEDS.items() if v["method"] == "DATE_ARCHIVE"}
PASSED_PAGED_FEEDS = {k: v for k, v in PASSED_FEEDS.items() if v["method"] == "PAGED"}

PARTIAL_FEEDS = feeds_where(summary_df["verdict"] == "PARTIAL")
LIVE_ONLY_FEEDS = feeds_where(summary_df["verdict"] == "LIVE_ONLY", with_method=False)

# 실패 사유 {feed_id: 사유}
FAILED_FEEDS = {
    row["feed_id"]: (
        f"수집 0건 ({row['method']})" if pd.notna(row["method"])
        else f"라이브 {row['live_status']}, 과거 접근 불가"
    )
    for _, row in summary_df[summary_df["verdict"] == "FAILED"].iterrows()
}

# 과거로 내려갈 수 있는 깊이 (통과/부분 피드)
def depth_of(feed_id):
    row = access_df.set_index("feed_id").loc[feed_id]
    method = row["method"]

    if method == "DATE_ARCHIVE":
        ok = [str(d) for d in DEPTH_PROBE_DATES if row.get(f"depth_{d}") == True]
        return f"아카이브 확인 날짜: {', '.join(ok) or '없음'}"

    return f"마지막 페이지 {int(row['last_page'])} → {row['paged_oldest_et']:%Y-%m-%d}까지"


FEED_DEPTH = {feed_id: depth_of(feed_id) for feed_id in {**PASSED_FEEDS, **PARTIAL_FEEDS}}

feed_lists = {
    "created_at": now_utc().isoformat(),
    "test_period": [TEST_START.isoformat(), TEST_END.isoformat()],
    "depth_probe_dates": [d.isoformat() for d in DEPTH_PROBE_DATES],
    "PASSED_DATE_ARCHIVE": PASSED_DATE_ARCHIVE_FEEDS,
    "PASSED_PAGED": PASSED_PAGED_FEEDS,
    "PARTIAL": PARTIAL_FEEDS,
    "LIVE_ONLY": LIVE_ONLY_FEEDS,
    "WAYBACK_ONLY": WAYBACK_ONLY_FEEDS,
    "FAILED": FAILED_FEEDS,
    "DEPTH": FEED_DEPTH,
}

FEED_LISTS_PATH.write_text(json.dumps(feed_lists, ensure_ascii=False, indent=2))

for key in ["PASSED_DATE_ARCHIVE", "PASSED_PAGED", "PARTIAL", "LIVE_ONLY", "WAYBACK_ONLY", "FAILED"]:
    print(f"{key:20s} {len(feed_lists[key]):2d}개: {list(feed_lists[key])}")

print()

for feed_id, depth in FEED_DEPTH.items():
    print(f"{feed_id:22s} {depth}")

PASSED_DATE_ARCHIVE   9개: ['ibd', 'nypost', 'thehill', 'nationalreview', 'motherjones', 'techcrunch', 'arstechnica', '9to5mac', 'semiengineering']
PASSED_PAGED          3개: ['fastcompany', 'wired', 'eetimes']
PARTIAL               0개: []
LIVE_ONLY            60개: ['cnbc_top', 'cnbc_business', 'cnbc_investing', 'cnbc_market_insider', 'cnbc_economy', 'cnbc_earnings', 'cnbc_finance', 'cnbc_tech', 'wsj_markets', 'wsj_business', 'wsj_economy', 'wsj_tech', 'wsj_world', 'marketwatch_top', 'marketwatch_bulletins', 'bloomberg_markets', 'bloomberg_economics', 'bloomberg_tech', 'seekingalpha_all', 'seekingalpha_currents', 'fool', 'nasdaq_markets', 'investing_news', 'investing_economy', 'marketbeat', 'thestreet', 'kiplinger', 'fortune', 'business_insider', 'foxbusiness_latest', 'foxbusiness_markets', 'foxbusiness_economy', 'cbs_moneywatch', 'nypost_business', 'forbes_business', 'forbes_innovation', 'qz', 'axios', 'semafor', 'nyt_business', 'nyt_economy', 'nyt_tech', 'nyt_home', 'latimes_business',

## 5. 실제 수집용 함수

통과 피드(`PASSED_FEEDS`)로 원하는 기간의 기사 목록을 모읍니다. 같은 언론사의 여러 섹션 피드에 같은 기사가 있으면
`(source, link)` 기준으로 합치고 섹션은 `sections`에 모아둡니다.

주의할 점
- `DATE_ARCHIVE` 피드는 4단계 깊이 결과에서 확인된 날짜까지는 같은 방식으로 수집 가능
- `PAGED` 피드는 마지막 페이지보다 과거로는 못 감
- 반환되는 것은 기사 목록(제목, 링크, 시각, 요약, RSS 본문 HTML `content`). 본문 텍스트는 8단계 `collect_news_with_body`에서 붙임

In [16]:
def collect_news(start, end, feeds=None):
    feeds = PASSED_FEEDS if feeds is None else feeds
    articles = []
    stats = {}

    for feed_id, info in feeds.items():
        try:
            rows, stat = collect_feed(feed_id, info, start, end)
        except Exception as e:
            rows, stat = [], {"error": f"{type(e).__name__}: {e}"}

        articles.extend(rows)
        stats[feed_id] = {"articles": len(rows), **stat}

    df = pd.DataFrame(articles)

    if df.empty:
        return df, stats

    df = (
        df.sort_values("published_utc")
        .groupby(["source", "link"], as_index=False)
        .agg(
            name=("name", "first"),
            sections=("section", lambda x: ", ".join(sorted(set(x)))),
            feed_ids=("feed_id", lambda x: ", ".join(sorted(set(x)))),
            method=("method", "first"),
            title=("title", "first"),
            published_utc=("published_utc", "first"),
            published_et=("published_et", "first"),
            summary=("summary", "first"),
            content=("content", "first"),
            author=("author", "first"),
        )
        .sort_values("published_utc")
        .reset_index(drop=True)
    )

    return df, stats


# 예시: 테스트 기간 첫날 하루 (3단계 캐시를 재사용하므로 대부분 네트워크 요청 없음)
news_df, news_stats = collect_news(TEST_START, TEST_START)

print(f"{TEST_START} 수집 기사 {len(news_df)}건 (언론사 {news_df['source'].nunique()}곳)")
news_df.groupby("name").size().sort_values(ascending=False).to_frame("articles")

2025-10-14 수집 기사 657건 (언론사 12곳)


,articles
name,
New York Post,246
The Hill,156
Fast Company,68
Investor's Business Daily,63
TechCrunch,28
9to5Mac,27
National Review,19
Semiconductor Engineering,15
WIRED,13


## 6. 눈으로 검수하기

피드별 표본 기사와 시간대별 기사 수 분포입니다. 특정 시간대가 통째로 비어 있으면 수집 누락을 의심하세요.

In [17]:
(
    articles_df
    .sort_values("published_utc")
    .groupby("feed_id")
    .head(2)[["feed_id", "method", "published_et", "title", "link"]]
)

,feed_id,method,published_et,title,link
376,nypost,DATE_ARCHIVE,2025-10-14 00:06:54-04:00,Mamdani’s biggest backer Democratic Socialists...,https://nypost.com/2025/10/14/us-news/zohran-m...
375,nypost,DATE_ARCHIVE,2025-10-14 00:28:43-04:00,"Cincinnati woman, 23, shot dead by friend duri...",https://nypost.com/2025/10/14/us-news/cincinna...
1204,9to5mac,DATE_ARCHIVE,2025-10-14 00:28:59-04:00,macOS 26.1 beta 3 changes polarizing Macintosh...,https://9to5mac.com/2025/10/13/macos-26-1-beta...
743,fastcompany,PAGED,2025-10-14 02:00:00-04:00,Most employers offer mental health care benefi...,https://www.fastcompany.com/91421154/most-empl...
50,ibd,DATE_ARCHIVE,2025-10-14 03:00:00-04:00,Cimpress Trying To Close In On Key Technical B...,https://www.investors.com/ibd-data-stories/cim...
45,ibd,DATE_ARCHIVE,2025-10-14 03:00:00-04:00,Brookfield Renewable Ptrs Getting Closer To Ke...,https://www.investors.com/ibd-data-stories/bro...
1271,eetimes,PAGED,2025-10-14 03:00:00-04:00,Diraq CEO Aims for Company’s First Quantum Com...,https://www.eetimes.com/diraq-ceo-aims-for-com...
1286,semiengineering,DATE_ARCHIVE,2025-10-14 03:01:26-04:00,Enabling In-Line Process Control for Hybrid Bo...,https://semiengineering.com/enabling-in-line-p...
1285,semiengineering,DATE_ARCHIVE,2025-10-14 03:01:30-04:00,Rethinking Security In Semiconductor Testing: ...,https://semiengineering.com/rethinking-securit...
1161,arstechnica,DATE_ARCHIVE,2025-10-14 03:05:16-04:00,SpaceX finally got exactly what it needed from...,https://arstechnica.com/space/2025/10/after-ye...


In [18]:
# 피드별 동부시간 시각대 기사 수 (테스트 기간 합계)
(
    articles_df
    .assign(hour_et=lambda d: d["published_et"].map(lambda t: t.hour))
    .pivot_table(index="feed_id", columns="hour_et", values="link", aggfunc="count", fill_value=0)
)

hour_et,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
feed_id,,,,,,,,,,,,,,,,,,,,,
9to5mac,1,0,0,0,0,0,1,4,5,9,...,6,2,7,3,2,2,1,0,2,1
arstechnica,0,0,0,1,0,0,0,0,0,4,...,3,2,4,3,0,0,0,0,0,0
eetimes,0,0,0,3,0,0,0,0,1,2,...,0,0,1,0,0,0,0,0,0,0
fastcompany,1,0,3,0,5,4,9,37,3,4,...,7,3,2,0,6,3,2,0,0,0
ibd,0,0,0,86,0,0,1,0,10,3,...,7,0,2,1,1,2,0,0,0,0
motherjones,0,0,0,0,0,0,2,1,0,0,...,2,0,0,4,1,0,0,0,0,0
nationalreview,1,1,0,0,0,0,7,1,4,4,...,4,3,4,3,2,1,1,0,0,0
nypost,5,10,5,6,0,3,14,18,11,20,...,21,35,49,35,34,19,26,23,15,9
semiengineering,0,0,0,18,0,0,0,0,0,0,...,2,0,0,1,1,0,0,0,0,0


## 7. 본문 수집 테스트

통과 피드(`PASSED_FEEDS`)의 테스트 기간 기사 중 피드별 최대 40건을 뽑아 **두 경로로 본문을 각각 구하고 서로 대조**합니다.

| 경로 | 방법 | 비고 |
|---|---|---|
| RSS 본문 | 피드의 `content:encoded`(Atom `content`) HTML → 문단 텍스트, 출처 푸터 제거 | 요청 없음. 전문인지 티저인지 원문과 비교해야 앎 |
| 원문 페이지 본문 | 기사 URL의 HTML → trafilatura 추출, 기준 본문(JSON-LD `articleBody` 또는 사이트별 본문 영역)으로 교차 확인 | `1_google_rss_source_test.ipynb`와 같은 방식·판정 + 보완 |

**검증 항목**
- 원문 페이지 판정: 차단 / HTTP 오류 / 영상·라이브 / 기사 아님 / 유료 / 본문 짧음 / 일부만 추출 / 성공
- 기준 본문 대조: trafilatura 본문과 기준 본문이 얼마나 겹치는지 (연속 5단어 묶음 기준)
  - 기준 본문 = 언론사가 페이지에 넣어둔 JSON-LD `articleBody`, 없으면 사이트별 본문 영역(`SITE_BODY_CONTAINERS`)의 문단
  - `ref_recall`: 기준 본문 중 trafilatura 본문에 들어있는 비율 → 낮으면 본문 누락 (0.8 미만이면 기준 본문을 대신 사용)
  - `ref_precision`: trafilatura 본문 중 기준 본문에 들어있는 비율 → 낮으면 광고·추천 기사 등 섞임
- RSS 대조: `rss_recall` = 원문 본문 중 RSS 본문에 들어있는 비율. 0.9 이상이면 RSS 본문을 전문으로 인정
- 같은 기사인지: RSS 제목 단어가 페이지 제목에 있는 비율, RSS 요약 문구가 본문에 있는 비율, RSS 시각과 페이지 작성 시각 차이
- 본문 오염: 뉴스레터·앱 홍보, 추천/관련 기사, 출처 푸터 문구가 본문에 남았는지

**피드 판정**
- `RSS_BODY`: 원문과 비교 가능한 기사의 80% 이상이 RSS 전문 → 수집 시 원문 요청 없이 RSS 본문 사용
- `PAGE_BODY`: 기사 페이지 중 원문 본문 추출 성공 80% 이상 → 수집 시 원문 페이지에서 추출
- `UNSTABLE` (50~80%), `NOT_USABLE` (50% 미만), `INSUFFICIENT_SAMPLE` (기사 페이지 8건 미만)

### 7-1. 본문 텍스트 유틸

In [19]:
HTML_DIR.mkdir(parents=True, exist_ok=True)

HEADING_TAGS = {"h2", "h3", "h4", "h5"}
BLOCK_TAGS = {"p", "li", "blockquote", "pre"} | HEADING_TAGS
DROP_TAGS = ["script", "style", "noscript", "figure", "figcaption", "iframe", "video", "audio"]


def has_junk_class(node):
    classes = (node.get("class") or "").lower().split()

    return any(c.startswith(JUNK_CLASS_PREFIXES) for c in classes)


def html_to_text(fragment, drop_junk=False):
    # 본문 HTML 조각 → 문단 텍스트 (문단마다 줄바꿈). 사진·캡션·스크립트는 제거
    # drop_junk=True면 광고·뉴스레터·관련 기사 등 class가 붙은 요소도 제거 (사이트 본문 영역용)
    if not isinstance(fragment, str) or not fragment.strip():
        return ""

    try:
        tree = lxml_html.fragment_fromstring(fragment, create_parent="div")
    except Exception:
        return ""

    for node in tree.xpath(" | ".join(f".//{tag}" for tag in DROP_TAGS)):
        node.drop_tree()

    if drop_junk:
        for node in [n for n in tree.iter() if isinstance(n.tag, str) and has_junk_class(n)]:
            node.drop_tree()

    # 문단 안에 든 문단(li 안의 p 등)은 바깥 것만 사용. (태그, 텍스트)
    blocks = []

    for node in tree.iter(*BLOCK_TAGS):
        if any(parent.tag in BLOCK_TAGS for parent in node.iterancestors()):
            continue

        text = " ".join(node.text_content().split())

        if drop_junk and text:
            link_chars = sum(len(" ".join(a.text_content().split())) for a in node.iter("a"))

            if link_chars / len(text) >= MAX_LINK_DENSITY:
                continue

        if text:
            blocks.append((node.tag, text))

    # 문단 태그 없이 <br>로만 나뉜 본문
    if not blocks:
        return " ".join(tree.text_content().split())

    # 본문 영역 끝에 문단 없이 남은 소제목 제거 (9to5Mac "Accessory deals on Amazon" 아래 상품 링크가 빠지고 제목만 남는 경우)
    if drop_junk:
        while blocks and blocks[-1][0] in HEADING_TAGS:
            blocks.pop()

    return "\n".join(text for _, text in blocks)


def clean_rss_body(text):
    # RSS 본문 끝 출처 문구 제거
    lines = [line for line in text.split("\n") if not RSS_FOOTER_PATTERN.search(line.strip())]

    return "\n".join(lines).strip()


def rss_body_of(content):
    return clean_rss_body(html_to_text(content))


def is_teaser(text):
    return bool(text) and bool(RSS_TEASER_PATTERN.search(text[-300:]))


def word_count(text):
    return len((text or "").split())


def words_of(text):
    return re.findall(r"[a-z0-9]+", unicodedata.normalize("NFKC", text or "").lower())


def shingles(text, n=SHINGLE_SIZE):
    words = words_of(text)

    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}


def share_in(part, whole, n=SHINGLE_SIZE):
    # part의 연속 n단어 묶음 중 whole에도 있는 비율 (part가 n단어보다 짧으면 None)
    part_set = shingles(part, n)

    if not part_set:
        return None

    return round(len(part_set & shingles(whole, n)) / len(part_set), 3)


def junk_hits(text):
    lines = [line.strip() for line in (text or "").split("\n")]

    return sorted(
        name
        for name, pattern in BODY_JUNK_PATTERNS.items()
        if any(pattern.search(line) for line in lines)
    )


def host_of(url):
    host = (urlparse(url or "").hostname or "").lower()

    return host[4:] if host.startswith("www.") else host


def is_excluded_section(url):
    host = host_of(url)
    target = host + urlparse(url).path
    rules = [
        rule
        for site, site_rules in SECTION_EXCLUDE_RULES.items()
        if host == site or host.endswith("." + site)
        for rule in site_rules
    ]

    return any(re.search(rule, target) for rule in rules)


def same_site(url_a, url_b):
    a, b = host_of(url_a), host_of(url_b)

    return a == b or a.endswith("." + b) or b.endswith("." + a)

### 7-2. 원문 HTML 다운로드 (캐시)

In [20]:
def html_cache_path(url):
    return HTML_DIR / f"{url_hash(url)}.html.gz"


def load_fetch_log():
    # 같은 URL은 마지막 기록이 우선
    return {record["url"]: record for record in load_jsonl(HTML_FETCH_LOG_PATH)}


def is_retryable_fetch(record):
    # 일시적 오류(연결 오류, 429, 5xx)만 재시도 대상
    return record["fetch_status"] == "REQUEST_ERROR" or record["http_status"] in {429, 500, 502, 503, 504}


def fetch_html(url):
    response, error = polite_get(url, delay=BODY_FETCH_DELAY, headers=HTML_HEADERS, retries=BODY_FETCH_RETRIES)

    record = {
        "url": url,
        "final_url": None,
        "http_status": None,
        "content_type": None,
        "fetch_status": "REQUEST_ERROR",
        "error": error,
        "html_saved": False,
        "fetched_at": now_utc().isoformat(),
    }

    if response is not None:
        content_type = response.headers.get("Content-Type", "").lower()

        record.update(
            final_url=response.url,
            http_status=response.status_code,
            content_type=content_type,
            fetch_status=(
                "HTTP_ERROR" if response.status_code != 200
                else "NOT_HTML" if "html" not in content_type
                else "OK"
            ),
        )

        # 오류 응답이라도 HTML이면 저장 (차단 페이지 확인용)
        if "html" in content_type:
            html_cache_path(url).write_bytes(gzip.compress(response.content))
            record["html_saved"] = True

    append_jsonl(HTML_FETCH_LOG_PATH, record)

    return record


def interleave_by_host(urls):
    # 같은 사이트에 연속 요청하지 않도록 호스트를 번갈아 정렬 (호스트별 대기 시간이 겹치게)
    queues = {}

    for url in dict.fromkeys(urls):
        queues.setdefault(host_of(url), []).append(url)

    ordered = []
    queues = list(queues.values())

    while any(queues):
        for queue in queues:
            if queue:
                ordered.append(queue.pop(0))

    return ordered


def is_host_blocking(record):
    return record["fetch_status"] == "REQUEST_ERROR" or record["http_status"] in {401, 403, 429}


def fetch_html_cached(urls, retry_errors=False, verbose=True):
    log = load_fetch_log()
    urls = interleave_by_host(urls)

    todo = [
        url for url in urls
        if url not in log or (retry_errors and is_retryable_fetch(log[url]))
    ]

    if verbose:
        print(f"URL {len(urls)}개 중 캐시 사용 {len(urls) - len(todo)}개, 새로 다운로드 {len(todo)}개", flush=True)

    # 사이트별 연속 차단 횟수. 한도를 넘으면 그 사이트는 이번 실행에서 더 요청하지 않음 (캐시에 남기지 않음)
    fail_streak = {}
    skipped = {}

    for i, url in enumerate(todo, start=1):
        host = host_of(url)

        if fail_streak.get(host, 0) >= BODY_HOST_FAIL_LIMIT:
            log[url] = {
                "url": url, "final_url": None, "http_status": None, "content_type": None,
                "fetch_status": "SKIPPED_HOST", "error": f"연속 {BODY_HOST_FAIL_LIMIT}회 차단으로 건너뜀",
                "html_saved": False, "fetched_at": None,
            }
            skipped[host] = skipped.get(host, 0) + 1
            continue

        log[url] = fetch_html(url)
        fail_streak[host] = fail_streak.get(host, 0) + 1 if is_host_blocking(log[url]) else 0

        if verbose and (i % 50 == 0 or i == len(todo)):
            print(f"  [{i}/{len(todo)}] {host} {log[url]['fetch_status']} ({log[url]['http_status']})", flush=True)

    if verbose and skipped:
        print("연속 차단으로 건너뛴 사이트:", skipped)

    return {url: log[url] for url in urls}

### 7-3. 원문 본문 추출 + 페이지 판정

`1_google_rss_source_test.ipynb` 4단계 함수를 그대로 가져와 두 가지만 바꿨습니다.
- 검색 대상일과 작성일 비교(`OUT_OF_WINDOW`) 대신 RSS 시각과 페이지 작성 시각 차이(`date_gap_hours`)를 기록 (RSS 날짜 아카이브가 이미 날짜를 보장하므로)
- 기사 판정에서 작성 시각 메타 필수 조건 제외 (Mother Jones는 JSON-LD·작성 시각 메타가 없지만 og:type=article인 정상 기사)

그리고 본문 선택을 보완했습니다.
1. 사이트별 본문 영역(`SITE_BODY_CONTAINERS`)이 등록된 사이트는 그 영역의 문단을 본문으로 사용 (rss_news의 매체별 SELECTORS 방식)
   - 영역 안의 광고·뉴스레터·관련 기사 class 요소와 링크 위주 문단(관련 기사·상품 목록)은 제외
   - trafilatura는 교차 확인용: Ars Technica는 본문이 광고를 사이에 두고 여러 덩어리로 나뉘어 trafilatura가 첫 덩어리만 가져가고,
     NY Post는 구독 홍보·뉴스레터 박스·연합 기사 머리글까지 본문에 넣는 것을 확인 (2026-11-05)
2. 미등록 사이트는 trafilatura 본문 사용. JSON-LD `articleBody`의 80% 미만만 들어 있으면 `articleBody` 사용 (rss_news 본문 시험의 Business Insider 방식)
3. 그래도 본문이 없으면 페이지 안 WordPress 글 데이터(JSON)의 `content.rendered` 사용 (National Review는 본문이 HTML 태그가 아니라 JSON에만 있음)

In [21]:
def read_cached_html(url):
    path = html_cache_path(url)

    if not path.exists():
        return None

    return decode_file(gzip.decompress(path.read_bytes()))


def parse_json_ld(tree):
    # 페이지의 JSON-LD 최상위 객체(+@graph)를 평탄화해서 리스트로 반환
    items = []

    for node in tree.xpath('//script[@type="application/ld+json"]'):
        try:
            data = json.loads(node.text_content().strip())
        except Exception:
            continue

        stack = [data]

        while stack:
            current = stack.pop()

            if isinstance(current, list):
                stack.extend(current)

            elif isinstance(current, dict):
                items.append(current)

                if "@graph" in current:
                    stack.append(current["@graph"])

    return items


def ld_types_of(item):
    types = item.get("@type", [])

    if isinstance(types, str):
        types = [types]

    return {t for t in types if isinstance(t, str)}


def meta_content(tree, attr, value, lower=True):
    found = tree.xpath(f'//meta[@{attr}="{value}"]/@content')

    if not found:
        return None

    return found[0].strip().lower() if lower else found[0].strip()


def parse_bool(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"

    return None


def to_news_tz(value):
    # 날짜 문자열 → 동부시간 datetime. 시간대 표기가 없으면 동부시간으로 간주
    if not isinstance(value, str) or not value.strip():
        return None

    try:
        dt = date_parser.parse(value.strip())
    except (ValueError, OverflowError):
        return None

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=NEWS_TIMEZONE)

    return dt.astimezone(NEWS_TIMEZONE)


def find_page_date(tree, article_items, fields):
    # fields 순서대로 찾아서 처음 해석되는 날짜
    for field in fields:
        kind, key = field.split(":", 1)

        if kind == "ld":
            values = [item.get(key) for item in article_items]
        else:
            values = tree.xpath(f'//meta[@{kind}="{key}"]/@content')

        for value in values:
            dt = to_news_tz(value)

            if dt:
                return dt

    return None


def site_container_body(tree, final_url):
    # SITE_BODY_CONTAINERS에 등록된 본문 영역의 문단 텍스트 (여러 개면 문서 순서대로 이어붙임)
    container = SITE_BODY_CONTAINERS.get(host_of(final_url))

    if not container:
        return ""

    tag, class_name = container
    nodes = tree.xpath(f"//{tag}[contains(concat(' ', normalize-space(@class), ' '), ' {class_name} ')]")
    node_ids = {id(node) for node in nodes}

    # 영역 안에 같은 영역이 또 있으면 바깥 것만
    nodes = [node for node in nodes if not any(id(parent) in node_ids for parent in node.iterancestors())]

    texts = [html_to_text(lxml_html.tostring(node, encoding="unicode"), drop_junk=True) for node in nodes]

    return "\n".join(t for t in texts if t)


# WordPress 글 데이터(JSON)의 본문 HTML: "content":{"rendered":"..."}
EMBEDDED_CONTENT_PATTERN = re.compile(r'"content":\{"rendered":("(?:[^"\\]|\\.)*")')


def embedded_json_body(raw_html):
    texts = []

    for match in EMBEDDED_CONTENT_PATTERN.finditer(raw_html):
        try:
            texts.append(html_to_text(json.loads(match.group(1))))
        except ValueError:
            continue

    return max(texts, key=len, default="")


def extract_page_features(raw_html, final_url):
    # lxml은 encoding 선언이 있는 str을 받지 못하므로 XML 선언 제거
    tree = lxml_html.fromstring(re.sub(r"^\s*<\?xml[^>]*\?>", "", raw_html))

    page_title = (
        meta_content(tree, "property", "og:title", lower=False)
        or (tree.findtext(".//title") or "").strip()
    )

    og_type = meta_content(tree, "property", "og:type")

    ld_items = parse_json_ld(tree)
    ld_types = set().union(*[ld_types_of(item) for item in ld_items]) if ld_items else set()
    article_items = [item for item in ld_items if ld_types_of(item) & ARTICLE_LD_TYPES]

    # 언론사가 페이지에 넣어둔 기사 본문 (가장 긴 것)
    ld_body = max(
        (" ".join(html.unescape(str(item.get("articleBody") or "")).split()) for item in article_items),
        key=len,
        default="",
    )

    ld_free_values = [
        parse_bool(item.get("isAccessibleForFree"))
        for item in article_items
        if "isAccessibleForFree" in item
    ]
    ld_is_free = None if not ld_free_values else all(v is not False for v in ld_free_values)

    published_dt = find_page_date(tree, article_items, PUBLISHED_DATE_FIELDS)

    path = urlparse(final_url).path
    url_is_video = bool(VIDEO_PATH_PATTERN.search(path) or VIDEO_HOST_PATTERN.search(host_of(final_url)))
    url_is_live = bool(LIVE_BLOG_PATH_PATTERN.search(path))

    title_lower = page_title.lower()
    html_lower = raw_html[:50000].lower()
    is_blocked = (
        any(phrase in title_lower for phrase in BLOCK_TITLE_PHRASES)
        or any(marker in html_lower for marker in BLOCK_HTML_MARKERS)
    )

    trafilatura_body = trafilatura_extract(
        raw_html,
        url=final_url,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
    )
    trafilatura_body = (trafilatura_body or "").strip()

    ld_words = word_count(ld_body)
    container_body = site_container_body(tree, final_url)

    # 교차 확인용 기준 본문: 언론사가 넣은 JSON-LD articleBody > 사이트별 본문 영역
    if ld_words >= MIN_BODY_WORDS:
        reference, reference_type = ld_body, "json_ld"
    elif word_count(container_body) >= MIN_BODY_WORDS:
        reference, reference_type = container_body, "container"
    else:
        reference, reference_type = "", None

    # 기준 본문 중 trafilatura 본문에 들어간 비율(누락 확인), trafilatura 본문 중 기준 본문에 있는 비율(오염 확인)
    ref_recall = share_in(reference, trafilatura_body) if reference and trafilatura_body else None
    ref_precision = share_in(trafilatura_body, reference) if reference and trafilatura_body else None

    body, body_method = trafilatura_body, "trafilatura"

    # 사이트별 본문 영역이 있으면 그것을 본문으로 사용 (trafilatura는 교차 확인용)
    #   trafilatura는 NY Post 구독 홍보·뉴스레터 박스·연합 기사 머리글, Ars Technica 사진 설명,
    #   9to5Mac 상품 목록까지 본문에 넣는 경우가 있었음 (2026-11-05 표본 380건 확인)
    if reference_type == "container":
        body, body_method = reference, reference_type

    # trafilatura가 JSON-LD 본문의 일부만 잡았으면 JSON-LD 본문 사용
    elif reference and (ref_recall or 0) < MIN_REF_COVERAGE and word_count(reference) > word_count(trafilatura_body):
        body, body_method = reference, reference_type

    # 본문이 HTML 태그가 아니라 페이지 안 JSON에만 있는 경우 (National Review)
    elif word_count(body) < MIN_BODY_WORDS:
        embedded = embedded_json_body(raw_html)

        if word_count(embedded) >= MIN_BODY_WORDS:
            body, body_method = embedded, "embedded_json"

    body_words = word_count(body)
    body_lower = body.lower()

    return {
        "page_title": page_title,
        "og_type": og_type,
        "ld_types": sorted(ld_types),
        "ld_has_article": bool(ld_types & ARTICLE_LD_TYPES),
        "ld_has_video": "VideoObject" in ld_types,
        "ld_has_liveblog": "LiveBlogPosting" in ld_types,
        "ld_is_free": ld_is_free,
        "ld_body": ld_body,
        "ld_words": ld_words,
        "page_published_et": published_dt,
        "url_is_video": url_is_video,
        "url_is_live": url_is_live,
        "is_blocked": is_blocked,
        "paywall_phrase": next((p for p in PAYWALL_PHRASES if p in body_lower), None),
        "body": body,
        "body_method": body_method,
        "trafilatura_words": word_count(trafilatura_body),
        "container_words": word_count(container_body),
        "reference_type": reference_type,
        "ref_recall": ref_recall,
        "ref_precision": ref_precision,
        "body_words": body_words,
        "ld_coverage": round(body_words / ld_words, 3) if ld_words else None,
    }


def classify_page(link, fetch, features):
    # 제외 구역은 URL만으로 판단 (실제 수집에서는 다운로드 전에 거름)
    if is_excluded_section(link):
        return "EXCLUDED_SECTION"

    if fetch is None:
        return "NOT_FETCHED"

    if fetch["fetch_status"] in {"REQUEST_ERROR", "NOT_HTML", "SKIPPED_HOST"}:
        return fetch["fetch_status"]

    if features is not None and features["is_blocked"]:
        return "BLOCKED"

    if fetch["fetch_status"] == "HTTP_ERROR":
        return f"HTTP_{fetch['http_status']}"

    if not same_site(fetch["final_url"], link):
        return "REDIRECTED_AWAY"

    if features is None:
        return "NOT_FETCHED"

    if (
        features["url_is_video"]
        or (features["og_type"] or "").startswith("video")
        or (features["ld_has_video"] and not features["ld_has_article"])
    ):
        return "VIDEO_PAGE"

    if features["url_is_live"] or features["ld_has_liveblog"]:
        return "LIVE_BLOG"

    # 기사 타입 메타데이터(JSON-LD 기사 또는 og:type=article)가 있어야 기사로 인정
    # 1_google_rss_source_test.ipynb는 og:type≠website와 작성 시각 메타도 요구했지만, 여기 링크는 RSS 기사라 작성 시각을 이미 알고 있고
    #   - Mother Jones: JSON-LD·작성 시각 메타가 없는 정상 기사 (og:type=article)
    #   - WIRED 리뷰: og:type=website지만 JSON-LD Review인 정상 기사
    # 가 있어 뺐음 (작성 시각 차이는 date_gap_hours로 따로 확인)
    if not (features["ld_has_article"] or features["og_type"] == "article"):
        return "NOT_ARTICLE"

    if not features["body"]:
        return "EXTRACT_FAILED"

    if features["paywall_phrase"]:
        return "PAYWALL"

    body_is_short = features["body_words"] < MIN_BODY_WORDS
    body_is_partial = features["ld_coverage"] is not None and features["ld_coverage"] < MIN_LD_COVERAGE

    # isAccessibleForFree=false여도 HTML에 본문 전체가 있는 경우가 많아 본문이 짧거나 일부일 때만 유료로 판정
    if features["ld_is_free"] is False and (body_is_short or body_is_partial):
        return "PAYWALL"

    if body_is_short:
        return "TOO_SHORT"

    if body_is_partial:
        return "PARTIAL_BODY"

    return "SUCCESS"


def analyze_page(link, fetch):
    # 저장된 HTML로 (판정, 특징, 분석 오류)
    features = None
    error = None

    if fetch and fetch["html_saved"]:
        raw_html = read_cached_html(link)

        if raw_html:
            try:
                features = extract_page_features(raw_html, fetch["final_url"])
            except Exception as e:
                error = f"{type(e).__name__}: {e}"

    status = "ANALYZE_ERROR" if error else classify_page(link, fetch, features)

    return status, features, error

### 7-4. 표본 추출 → 다운로드 → 두 경로 본문 대조

In [22]:
def sample_articles(articles_df, feeds, per_feed=BODY_SAMPLE_PER_FEED, seed=BODY_SAMPLE_SEED):
    groups = []

    for feed_id in feeds:
        group = articles_df[articles_df["feed_id"] == feed_id].drop_duplicates("link")
        groups.append(group.sample(min(per_feed, len(group)), random_state=seed))

    return pd.concat(groups, ignore_index=True)


def check_body(article, fetch):
    status, features, error = analyze_page(article["link"], fetch)
    features = features or {}

    page_body = features.get("body", "") if status == "SUCCESS" else ""
    rss_body = rss_body_of(article.get("content"))
    summary_text = html_to_text(article.get("summary")) or article.get("summary") or ""

    # RSS 본문이 원문 본문을 얼마나 덮는지 (원문 추출이 성공한 기사만 비교 가능)
    rss_recall = share_in(page_body, rss_body) if page_body and rss_body else None
    rss_full = rss_recall is not None and rss_recall >= RSS_FULL_COVERAGE

    if rss_full:
        body, body_source = rss_body, "RSS"
    elif page_body:
        body, body_source = page_body, "PAGE"
    else:
        body, body_source = "", None

    published = features.get("page_published_et")
    date_gap = (
        round((article["published_utc"] - published).total_seconds() / 3600, 2)
        if published is not None else None
    )

    return {
        "feed_id": article["feed_id"],
        "name": article["name"],
        "title": article["title"],
        "link": article["link"],
        "published_et": article["published_et"],
        "final_url": fetch["final_url"] if fetch else None,
        "http_status": fetch["http_status"] if fetch else None,
        "status": status,
        "analyze_error": error,
        "page_title": features.get("page_title"),
        "body_method": features.get("body_method"),
        "page_words": word_count(page_body),
        "trafilatura_words": features.get("trafilatura_words"),
        "ld_words": features.get("ld_words"),
        "container_words": features.get("container_words"),
        "reference_type": features.get("reference_type"),
        "ref_recall": features.get("ref_recall"),
        "ref_precision": features.get("ref_precision"),
        "rss_words": word_count(rss_body),
        "rss_teaser": is_teaser(rss_body),
        "rss_recall": rss_recall,
        "rss_precision": share_in(rss_body, page_body) if page_body and rss_body else None,
        "rss_full": rss_full,
        "title_match": share_in(article["title"], features.get("page_title") or "", n=1),
        "summary_in_body": share_in(summary_text, body, n=3) if body else None,
        "date_gap_hours": date_gap,
        "body_source": body_source,
        # 최종 본문을 어디서 얻었는지: rss / trafilatura / json_ld / container / embedded_json
        "final_method": "rss" if body_source == "RSS" else features.get("body_method") if body_source == "PAGE" else None,
        "body_words": word_count(body),
        "junk": junk_hits(body),
        "body": body,
        "rss_body": rss_body,
        "page_body": page_body,
    }


body_sample_df = sample_articles(articles_df, PASSED_FEEDS)
print(f"표본 기사 {len(body_sample_df)}건 (피드 {body_sample_df['feed_id'].nunique()}개)")

fetch_map = fetch_html_cached(
    [link for link in body_sample_df["link"] if not is_excluded_section(link)],
    retry_errors=True,
)

body_check_df = pd.DataFrame([
    check_body(article, fetch_map.get(article["link"]))
    for article in body_sample_df.to_dict("records")
])
body_check_df.to_parquet(BODY_CHECK_PATH, index=False)

print(f"저장: {BODY_CHECK_PATH} ({len(body_check_df)}건)")

pd.crosstab(body_check_df["feed_id"], body_check_df["status"], margins=True)

표본 기사 380건 (피드 12개)
URL 379개 중 캐시 사용 302개, 새로 다운로드 77개


연속 차단으로 건너뛴 사이트: {'eetimes.com': 6, 'fastcompany.com': 31, 'thehill.com': 31}


저장: newsrss_test/body_check.parquet (380건)


status,BLOCKED,EXCLUDED_SECTION,LIVE_BLOG,PAYWALL,REQUEST_ERROR,SKIPPED_HOST,SUCCESS,TOO_SHORT,VIDEO_PAGE,All
feed_id,,,,,,,,,,
9to5mac,0,0,0,0,0,0,40,0,0,40
arstechnica,0,0,0,0,0,0,26,0,0,26
eetimes,0,0,0,0,3,6,0,0,0,9
fastcompany,9,0,0,0,0,31,0,0,0,40
ibd,0,0,1,39,0,0,0,0,0,40
motherjones,0,0,0,0,0,0,15,0,0,15
nationalreview,0,0,0,0,0,0,38,0,2,40
nypost,0,1,0,0,0,0,39,0,0,40
semiengineering,0,0,0,0,0,0,20,3,0,23


### 7-5. 검증 지표 + 피드 판정

| 열 | 의미 |
|---|---|
| `article_pages` | 표본 중 기사 페이지 수 (영상·라이브·기사 아닌 페이지 제외) |
| `page_success_rate` | 기사 페이지 중 원문 본문 추출 성공 비율 |
| `ref_compared`, `ref_recall_med`, `ref_precision_med` | 기준 본문과 비교한 기사 수, trafilatura 본문과의 겹침 비율 중앙값 |
| `rss_body_rate`, `rss_teaser_rate` | 표본 중 RSS 본문이 80단어 이상인 비율, 티저 문구로 끝나는 비율 |
| `rss_comparable`, `rss_full_rate`, `rss_recall_med` | 원문과 비교 가능한 기사 수, 그중 RSS가 전문인 비율, 원문 대비 RSS 커버율 중앙값 |
| `body_ok_rate` | 기사 페이지 중 최종 본문(RSS 전문 또는 원문)을 얻은 비율 |
| `title_match_med`, `title_mismatch` | RSS 제목 단어가 페이지 제목에 들어있는 비율 중앙값, 절반 미만인 기사 수 (같은 기사인지 확인) |
| `summary_in_body_med` | RSS 요약의 연속 3단어 묶음이 최종 본문에 들어있는 비율 중앙값 (요약이 리드 문장이면 높고, 별도 부제면 낮음) |
| `date_gap_over_24h` | RSS 시각과 페이지 작성 시각이 24시간 넘게 차이 나는 기사 수 (갱신형 기사·재게시 의심) |
| `junk_rate` | 최종 본문에 홍보·추천·푸터 문구가 남은 기사 비율 |

In [23]:
def summarize_bodies(check_df):
    df = check_df.assign(
        is_article=lambda d: ~d["status"].isin(FILTERABLE_STATUSES),
        is_success=lambda d: d["status"] == "SUCCESS",
        has_rss_body=lambda d: d["rss_words"] >= MIN_BODY_WORDS,
        rss_comparable=lambda d: d["rss_recall"].notna(),
        has_body=lambda d: d["body_words"] > 0,
        has_junk=lambda d: d["junk"].str.len() > 0,
        title_mismatch=lambda d: d["title_match"] < 0.5,
        date_gap_over_24h=lambda d: d["date_gap_hours"].abs() > 24,
    )
    articles = df[df["is_article"]]

    summary = df.groupby("feed_id").agg(
        name=("name", "first"),
        sampled=("link", "size"),
        rss_body_rate=("has_rss_body", "mean"),
        rss_teaser_rate=("rss_teaser", "mean"),
        rss_comparable=("rss_comparable", "sum"),
        rss_recall_med=("rss_recall", "median"),
        title_match_med=("title_match", "median"),
        title_mismatch=("title_mismatch", "sum"),
        summary_in_body_med=("summary_in_body", "median"),
        date_gap_over_24h=("date_gap_over_24h", "sum"),
    ).join(
        articles.groupby("feed_id").agg(
            article_pages=("link", "size"),
            page_success_rate=("is_success", "mean"),
            page_words_med=("page_words", lambda x: x[x > 0].median()),
            ref_compared=("ref_recall", "count"),
            ref_recall_med=("ref_recall", "median"),
            ref_precision_med=("ref_precision", "median"),
            body_ok_rate=("has_body", "mean"),
            junk_rate=("has_junk", lambda x: x[articles.loc[x.index, "has_body"]].mean()),
        )
    )

    summary["rss_full_rate"] = (
        df[df["rss_comparable"]].groupby("feed_id")["rss_full"].mean()
    )

    failures = articles[articles["status"] != "SUCCESS"]
    summary["main_fail_reason"] = failures.groupby("feed_id")["status"].agg(lambda x: x.value_counts().index[0])

    def body_verdict(row):
        if not row["article_pages"] >= MIN_BODY_ARTICLES:
            return "INSUFFICIENT_SAMPLE"

        if row["rss_comparable"] >= MIN_RSS_COMPARABLE and row["rss_full_rate"] >= BODY_USABLE_RATE:
            return "RSS_BODY"

        if row["page_success_rate"] >= BODY_USABLE_RATE:
            return "PAGE_BODY"

        if row["page_success_rate"] >= BODY_UNSTABLE_RATE:
            return "UNSTABLE"

        return "NOT_USABLE"

    summary["body_verdict"] = summary.apply(body_verdict, axis=1)

    return summary.reset_index()


body_summary_df = summarize_bodies(body_check_df)
body_summary_df.to_csv(BODY_SUMMARY_PATH, index=False, encoding="utf-8-sig")

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)

body_summary_df[[
    "feed_id", "body_verdict", "sampled", "article_pages", "page_success_rate", "page_words_med",
    "ref_compared", "ref_recall_med", "ref_precision_med",
    "rss_body_rate", "rss_teaser_rate", "rss_comparable", "rss_full_rate", "rss_recall_med",
    "body_ok_rate", "title_match_med", "title_mismatch", "summary_in_body_med", "date_gap_over_24h", "junk_rate", "main_fail_reason",
]].round(3).sort_values(["body_verdict", "feed_id"])

,feed_id,body_verdict,sampled,article_pages,page_success_rate,page_words_med,ref_compared,ref_recall_med,ref_precision_med,rss_body_rate,rss_teaser_rate,rss_comparable,rss_full_rate,rss_recall_med,body_ok_rate,title_match_med,title_mismatch,summary_in_body_med,date_gap_over_24h,junk_rate,main_fail_reason
2,eetimes,NOT_USABLE,9,9,0.00,NaN,0,NaN,NaN,0.000,0.000,0,NaN,NaN,0.00,0.0,9,NaN,0,NaN,SKIPPED_HOST
3,fastcompany,NOT_USABLE,40,40,0.00,NaN,0,NaN,NaN,0.000,0.000,0,NaN,NaN,0.00,0.0,40,NaN,0,NaN,SKIPPED_HOST
4,ibd,NOT_USABLE,40,39,0.00,NaN,0,NaN,NaN,0.000,0.000,0,NaN,NaN,0.00,1.0,2,NaN,0,NaN,PAYWALL
10,thehill,NOT_USABLE,40,40,0.00,NaN,0,NaN,NaN,0.000,0.000,0,NaN,NaN,0.00,0.0,40,NaN,0,NaN,SKIPPED_HOST
0,9to5mac,PAGE_BODY,40,40,1.00,297.0,40,1.000,1.000,0.000,0.000,0,NaN,NaN,1.00,1.0,1,1.000,0,0.000,NaN
1,arstechnica,PAGE_BODY,26,26,1.00,647.0,26,0.487,1.000,1.000,0.000,26,0.00,0.256,1.00,1.0,0,0.129,0,0.000,NaN
6,nationalreview,PAGE_BODY,40,38,1.00,532.5,0,NaN,NaN,0.000,0.000,29,0.00,0.000,1.00,1.0,0,0.534,0,0.000,NaN
7,nypost,PAGE_BODY,40,39,1.00,498.0,39,1.000,1.000,0.000,0.025,39,0.00,0.019,1.00,1.0,1,0.842,0,0.000,NaN
9,techcrunch,PAGE_BODY,40,39,1.00,429.0,39,1.000,1.000,0.000,0.000,0,NaN,NaN,1.00,1.0,0,0.185,0,0.000,NaN
11,wired,PAGE_BODY,27,26,1.00,1046.0,26,1.000,0.945,0.000,0.000,0,NaN,NaN,1.00,1.0,5,0.120,3,0.115,NaN


In [24]:
# 최종 본문을 어디서 얻었는지 (rss / trafilatura / json_ld / container / embedded_json)
pd.crosstab(body_check_df["feed_id"], body_check_df["final_method"].fillna("(본문 없음)"), margins=True)

final_method,(본문 없음),container,embedded_json,json_ld,rss,trafilatura,All
feed_id,,,,,,,
9to5mac,0,40,0,0,0,0,40
arstechnica,0,26,0,0,0,0,26
eetimes,9,0,0,0,0,0,9
fastcompany,40,0,0,0,0,0,40
ibd,40,0,0,0,0,0,40
motherjones,0,0,0,0,15,0,15
nationalreview,2,0,38,0,0,0,40
nypost,1,39,0,0,0,0,40
semiengineering,3,0,0,0,19,1,23


In [25]:
# 본문 오염 문구 종류별 기사 수 (최종 본문 기준)
(
    body_check_df[body_check_df["body_words"] > 0]
    .explode("junk")
    .dropna(subset=["junk"])
    .pivot_table(index="feed_id", columns="junk", values="link", aggfunc="count", fill_value=0)
)

junk,related,rss_footer
feed_id,,
motherjones,1,0
semiengineering,4,0
wired,0,3


In [26]:
# ── 변수 정리: 본문 판정 ──
def body_feeds_where(verdict):
    ids = body_summary_df.loc[body_summary_df["body_verdict"] == verdict, "feed_id"]

    return {
        feed_id: {**PASSED_FEEDS[feed_id], "body_method": verdict}
        for feed_id in PASSED_FEEDS
        if feed_id in set(ids)
    }


RSS_BODY_FEEDS = body_feeds_where("RSS_BODY")         # RSS 본문이 전문 → 원문 요청 없이 사용
PAGE_BODY_FEEDS = body_feeds_where("PAGE_BODY")       # 원문 페이지에서 본문 추출
BODY_UNSTABLE_FEEDS = body_feeds_where("UNSTABLE")    # 원문 추출이 절반 이상만 성공
BODY_NOT_USABLE_FEEDS = body_feeds_where("NOT_USABLE")
BODY_INSUFFICIENT_FEEDS = body_feeds_where("INSUFFICIENT_SAMPLE")

# 본문 수집에 쓸 피드 {feed_id: RSS_BODY / PAGE_BODY / UNSTABLE}
BODY_METHOD_BY_FEED = {
    feed_id: info["body_method"]
    for feed_id, info in {**RSS_BODY_FEEDS, **PAGE_BODY_FEEDS, **BODY_UNSTABLE_FEEDS}.items()
}

# 원문 추출 실패 주 사유 {feed_id: 상태값}
BODY_FAIL_REASON = (
    body_summary_df.dropna(subset=["main_fail_reason"])
    .set_index("feed_id")["main_fail_reason"]
    .to_dict()
)

body_lists = {
    "created_at": now_utc().isoformat(),
    "test_period": [TEST_START.isoformat(), TEST_END.isoformat()],
    "sample_per_feed": BODY_SAMPLE_PER_FEED,
    "RSS_BODY": RSS_BODY_FEEDS,
    "PAGE_BODY": PAGE_BODY_FEEDS,
    "UNSTABLE": BODY_UNSTABLE_FEEDS,
    "NOT_USABLE": BODY_NOT_USABLE_FEEDS,
    "INSUFFICIENT_SAMPLE": BODY_INSUFFICIENT_FEEDS,
    "FAIL_REASON": BODY_FAIL_REASON,
}

BODY_LISTS_PATH.write_text(json.dumps(body_lists, ensure_ascii=False, indent=2))

for key in ["RSS_BODY", "PAGE_BODY", "UNSTABLE", "NOT_USABLE", "INSUFFICIENT_SAMPLE"]:
    print(f"{key:20s} {len(body_lists[key]):2d}개: {list(body_lists[key])}")

print("실패 주 사유:", BODY_FAIL_REASON)

RSS_BODY              2개: ['motherjones', 'semiengineering']
PAGE_BODY             6개: ['nypost', 'nationalreview', 'techcrunch', 'arstechnica', 'wired', '9to5mac']
UNSTABLE              0개: []
NOT_USABLE            4개: ['ibd', 'fastcompany', 'thehill', 'eetimes']
INSUFFICIENT_SAMPLE   0개: []
실패 주 사유: {'eetimes': 'SKIPPED_HOST', 'fastcompany': 'SKIPPED_HOST', 'ibd': 'PAYWALL', 'semiengineering': 'TOO_SHORT', 'thehill': 'SKIPPED_HOST'}


## 8. 본문까지 붙이는 수집 함수

`collect_news`로 기사 목록을 받은 뒤 7단계 판정에 따라 본문을 붙입니다.
- `RSS_BODY` 피드: RSS 본문 사용 (요청 없음). RSS 본문이 짧거나 티저면 원문 페이지로 대체
- `PAGE_BODY`, `UNSTABLE` 피드: 원문 HTML을 받아(캐시) 추출. 판정이 `SUCCESS`인 경우만 본문으로 채택
- 제외 구역(`SECTION_EXCLUDE_RULES`, 예: NY Post 도박 프로모션) 기사는 요청하지 않음(`EXCLUDED_SECTION`)
- 그 외 피드: 본문 없음(`NOT_COLLECTED`). rss_news 팀 기준과 같이 요약만 있는 기사는 본문 대신 쓰지 않음

In [27]:
def attach_bodies(news_df, body_methods=None, retry_errors=False):
    body_methods = BODY_METHOD_BY_FEED if body_methods is None else body_methods
    rows = news_df.to_dict("records")
    plans = []

    # 1) RSS 본문으로 끝나는 기사와 원문이 필요한 기사 구분
    for row in rows:
        feed_id = row["feed_ids"].split(", ")[0]
        method = body_methods.get(feed_id)
        rss_body = rss_body_of(row.get("content"))

        if method == "RSS_BODY" and word_count(rss_body) >= MIN_BODY_WORDS and not is_teaser(rss_body):
            plans.append(("RSS", rss_body))
        elif is_excluded_section(row["link"]):
            plans.append(("EXCLUDED", None))
        elif method in {"RSS_BODY", "PAGE_BODY", "UNSTABLE"}:
            plans.append(("PAGE", None))
        else:
            plans.append((None, None))

    # 2) 원문 다운로드 (캐시)
    page_links = [row["link"] for row, (plan, _) in zip(rows, plans) if plan == "PAGE"]
    fetch_map = fetch_html_cached(page_links, retry_errors=retry_errors) if page_links else {}

    # 3) 본문 채우기
    results = []

    for row, (plan, text) in zip(rows, plans):
        if plan == "RSS":
            body, source, status = text, "RSS", "SUCCESS"
        elif plan == "PAGE":
            status, features, _ = analyze_page(row["link"], fetch_map.get(row["link"]))
            body, source = (features["body"], "PAGE") if status == "SUCCESS" else ("", None)
        elif plan == "EXCLUDED":
            body, source, status = "", None, "EXCLUDED_SECTION"
        else:
            body, source, status = "", None, "NOT_COLLECTED"

        results.append({"body": body, "body_source": source, "body_status": status, "body_words": word_count(body)})

    return pd.concat([news_df.reset_index(drop=True), pd.DataFrame(results)], axis=1)


def collect_news_with_body(start, end, feeds=None):
    news_df, stats = collect_news(start, end, feeds)

    if news_df.empty:
        return news_df, stats

    return attach_bodies(news_df), stats


# 예시: 5단계에서 받은 하루치 기사 중 언론사별 3건만 본문까지
demo_df = attach_bodies(news_df.groupby("name").head(3))

print(demo_df["body_status"].value_counts().to_string())
demo_df[["name", "title", "body_source", "body_status", "body_words"]]

URL 18개 중 캐시 사용 18개, 새로 다운로드 0개


body_status
SUCCESS          24
NOT_COLLECTED    12


,name,title,body_source,body_status,body_words
0,New York Post,Mamdani’s biggest backer Democratic Socialists...,PAGE,SUCCESS,685
1,New York Post,"Cincinnati woman, 23, shot dead by friend duri...",PAGE,SUCCESS,316
2,9to5Mac,macOS 26.1 beta 3 changes polarizing Macintosh...,PAGE,SUCCESS,294
3,New York Post,Trump says ‘we’ll have to see’ on one-state or...,PAGE,SUCCESS,333
4,Fast Company,Most employers offer mental health care benefi...,None,NOT_COLLECTED,0
5,Investor's Business Daily,Cellebrite DI Sees RS Rating Climb To 71,None,NOT_COLLECTED,0
6,Investor's Business Daily,Canadian Solar Earns Relative Strength Rating ...,None,NOT_COLLECTED,0
7,Investor's Business Daily,Calumet Sees IBD RS Rating Rise To 77,None,NOT_COLLECTED,0
8,EE Times,Diraq CEO Aims for Company’s First Quantum Com...,None,NOT_COLLECTED,0
9,Semiconductor Engineering,Enabling In-Line Process Control for Hybrid Bo...,RSS,SUCCESS,117


## 9. 본문 눈으로 검수하기

자동 지표가 놓치는 문제(본문 앞뒤에 붙은 다른 기사, 사진 설명, 홍보 문구)를 보기 위해 본문의 **처음과 끝**을 같이 봅니다.
피드 × 판정별 최대 3건은 `body_review.csv`로도 저장합니다.

In [28]:
def show_bodies(feed_id, status=None, n=2, chars=400):
    rows = body_check_df[body_check_df["feed_id"] == feed_id]

    if status:
        rows = rows[rows["status"] == status]

    for row in rows.head(n).to_dict("records"):
        body = row["body"] or ""
        print(f"[{row['status']} / 최종 본문 {row['final_method']}] {row['title']}")
        print(f"URL: {row['final_url'] or row['link']}")
        print(
            f"원문 {row['page_words']}단어 | trafilatura {row['trafilatura_words']} | JSON-LD {row['ld_words']} "
            f"| 본문 영역 {row['container_words']} | 기준 대비 recall {row['ref_recall']} precision {row['ref_precision']} "
            f"| RSS {row['rss_words']}단어 (원문 대비 {row['rss_recall']}) | 오염 {row['junk']}"
        )
        print("── 처음 ──")
        print(body[:chars])
        print("── 끝 ──")
        print(body[-chars:] if len(body) > chars else "")
        print("\n" + "=" * 100 + "\n")


review_df = (
    body_check_df
    .groupby(["feed_id", "status"], group_keys=False)
    .head(3)
    .assign(
        body_head=lambda d: d["body"].fillna("").str[:500],
        body_tail=lambda d: d["body"].fillna("").str[-500:],
    )
    [["feed_id", "status", "body_source", "title", "final_url", "page_words", "rss_words", "rss_recall",
      "reference_type", "ref_recall", "ref_precision", "junk", "body_head", "body_tail"]]
    .sort_values(["feed_id", "status"])
)
review_df.to_csv(BODY_REVIEW_PATH, index=False, encoding="utf-8-sig")

print(f"저장: {BODY_REVIEW_PATH} ({len(review_df)}건)")

저장: newsrss_test/body_review.csv (54건)


In [29]:
# 피드마다 성공 기사 1건씩 처음/끝 확인
for feed_id in PASSED_FEEDS:
    show_bodies(feed_id, status="SUCCESS", n=1, chars=300)

[SUCCESS / 최종 본문 container] Darren Waller blames Giants for his forgettable season before brief retirement
URL: https://nypost.com/2025/10/14/sports/darren-waller-blames-giants-for-his-forgettable-season-before-retirement/
원문 349단어 | trafilatura 375.0 | JSON-LD 0.0 | 본문 영역 349.0 | 기준 대비 recall 1.0 precision 0.932 | RSS 19단어 (원문 대비 0.003) | 오염 []
── 처음 ──
The Giants hoped Darren Waller could be a major weapon for Daniel Jones when they acquired him from the Raiders after their surprise playoff run in 2022.
But the former Pro Bowl tight end with a history of injury problems did not have the desired impact, accruing 552 yards and just one touchdown in 
── 끝 ──
eld his rights, and they got a 2026 sixth-round pick from Miami for Waller and a 2027 seventh-rounder.
Waller said he decided in June he was going to play again and didn’t know if he would have this much of an impact so soon.
“It surprises me a little bit,” he said. “I don’t think the body forgets.”


[SUCCESS / 최종 본문 embedded_json]

In [30]:
# 실패 기사 확인 (피드별 1건)
for feed_id, reason in BODY_FAIL_REASON.items():
    show_bodies(feed_id, status=reason, n=1, chars=300)

[SKIPPED_HOST / 최종 본문 None] Benchmarking AI Processors: Measuring What Matters
URL: https://www.eetimes.com/benchmarking-ai-processors-measuring-what-matters/
원문 0단어 | trafilatura nan | JSON-LD nan | 본문 영역 nan | 기준 대비 recall nan precision nan | RSS 28단어 (원문 대비 nan) | 오염 []
── 처음 ──

── 끝 ──



[SKIPPED_HOST / 최종 본문 None] How Gen Z plans to own homes no one thought they could
URL: https://www.fastcompany.com/91421994/how-gen-z-plans-to-own-homes-no-one-thought-they-could
원문 0단어 | trafilatura nan | JSON-LD nan | 본문 영역 nan | 기준 대비 recall nan precision nan | RSS 0단어 (원문 대비 nan) | 오염 []
── 처음 ──

── 끝 ──



[PAYWALL / 최종 본문 None] Stratasys Gets RS Rating Upgrade
URL: https://www.investors.com/ibd-data-stories/stratasys-gets-rs-rating-upgrade/
원문 0단어 | trafilatura 70.0 | JSON-LD 0.0 | 본문 영역 0.0 | 기준 대비 recall nan precision nan | RSS 0단어 (원문 대비 nan) | 오염 []
── 처음 ──

── 끝 ──



[TOO_SHORT / 최종 본문 None] EBook: Helping To Realize Chiplet Ambitions
URL: https://semiengineering.com/ebook-helping-